# Justification analysis: linking what the models *say* to what they *do*

## The research questions this notebook serves

> **RQ1 -- Deductive reasoning.** To what extent do LLMs reason deductively about
> explicit game rules when judging a social-deduction game?
>
> **RQ2 -- Social influence.** To what extent are their judgements driven instead by
> social pressure in the dialogue -- who is being accused, who is talked about?

Three models spanning a wide range of rule-reasoning proficiency (**2B**, **4B**, **31B**;
proficiency measured independently by the `preliminary_eval` questionnaire) read the same
191 real ONUW transcripts and vote for the Werewolf with a short written justification.

## Why this notebook exists

Earlier stages established two facts that, together, create a puzzle:

- **Vote accuracy is flat** (~0.41) across the whole proficiency range. Rule competence
  does not convert into task accuracy.
- **The vote is largely predicted by social salience** -- a surrogate model reaches
  0.41-0.56 top-1 (random 0.23) using essentially two features: how often a player was
  *accused* (`werewolf_count`) and whether they *claimed* to be the Werewolf
  (`claims_werewolf`). Speaker-side persuasion technique is close to null.

If rule competence exists but does not show up in accuracy, it must be looked for in
*how* the models arrive at their votes. This notebook therefore joins every justification
to its own vote and to the game's ground truth, and asks what the language reveals.

## Roadmap -- what each analysis tests and which RQ it serves

| # | Analysis | Serves | Question it answers |
|---|---|---|---|
| 1 | Markers x surrogate-predictability | RQ1 + RQ2 | Is "reasoning language" denser when a vote is *not* explained by social salience? |
| 2 | Markers x game class | RQ2 | When a model defies the crowd, does its language change? |
| 3 | Rule-reference lexicon | RQ1 | Do models differ in how much rule *vocabulary* they use? |
| 4 | Mention-check | RQ2 | When a vote follows social salience, does the text admit it? |
| **5** | **Ground-truth swap tracking** | **RQ1** | **When a role actually changed, does the justification get it right?** |
| **6** | **Ground-truth contradiction flagging** | **RQ1** | **When a player really did contradict themselves, does the model catch it?** |
| 7 | Salience-reliance gap | RQ1 + RQ2 | Do right and wrong votes come from the same mechanism? **⚠ inconclusive -- confounded, see §7** |

**Analyses 1-4 measure language. Analyses 5-7 measure correctness against ground
truth.** That progression is deliberate and is the methodological spine of the thesis:
counting words that *sound like* reasoning turns out not to work (see Analysis 3's
retracted lexicon and Analysis 1's null), so deduction is instead established by checking
whether the model's stated claims are *true of the actual game state*.

## Two controls used repeatedly

- **Chance-adjusted check (Analyses 5, 6).** A presence-based metric ("is the role named
  anywhere?") could reward a model that simply names many roles for stylistic reasons.
  Each check computes the exact probability of a coincidental match under that "just
  verbose" null (hypergeometric, given how many *irrelevant* roles that specific
  justification happens to mention) and reports the **excess** over it.
- **Discourse-connective precision layer (Analyses 5, 6).** Checks whether the two
  role mentions are actually *linked* ("started as X **but** ended as Y") rather than
  being two unrelated mentions in the same paragraph. Reuses the published DiMLex
  lexicon -- no new vocabulary is invented.

## Conventions (apply throughout, stated once here)

- All marker/lexicon counts are normalized **per 100 words**.
- Aggregation is **per-instance -> per-game mean -> mean across games**
  (equal game weighting). This matches the *extension* section of the
  existing DiMLex notebook, not its main body's run-level
  mean +/- SD-across-3-runs convention -- a deliberate, documented choice
  since the two sections of that notebook disagree.
- Stochastic runs (`run_1/2/3`, `decoding=="stochastic"`) are pooled as an
  empirical distribution over 3 T=1 samples (up to 3 instances/game feed the
  per-game mean). Greedy (`run_label=="greedy_t0"`) is always reported
  separately, never pooled with stochastic.
- Every number ships with `n_games` and `n_instances`. Every contrast ships a
  5000-draw 95% percentile bootstrap CI (games as the resampling unit, seeded
  `default_rng(0)`). Any CI spanning 0 is reported but flagged
  `inconclusive_at_this_n`, never narrated as a real difference.
- Interpretable methods only. Marker/lexicon mentions are not proof of rule
  use or reasoning; the mention-check in Analysis 4 is not proof of causal
  faithfulness. Caveats are repeated inline per section.

The DiMLex marker-matching work referenced throughout lives in the sibling
notebook `1_dimlex_justification_analysis.ipynb`. This notebook reads only
the already-saved `src/justification_analysis/dimlex_marker_assignments.csv`,
so it does not depend on that notebook being open or re-run.


In [1]:
from pathlib import Path
import json
import re
import warnings

import numpy as np
import pandas as pd
from scipy.stats import hypergeom

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 140)


def find_repo_root(start=None, repo_name="masters_thesis_sdg"):
    current = (start or Path.cwd()).resolve()
    while True:
        if current.name == repo_name:
            return current
        if current.parent == current:
            raise FileNotFoundError(f"Could not find repo root {repo_name!r} above {Path.cwd()}")
        current = current.parent


REPO_ROOT = find_repo_root()
ANALYSIS_ROOT = REPO_ROOT / "analysis"
DATA_PROCESSED = REPO_ROOT / "data" / "processed" / "lai2023"

MODEL_FOLDER_PATTERNS = {
    "2B": "*gemma-4-E2B*",
    "4B": "*gemma-4-E4B*",
    "31B": "*gemma-4-31B*",
}
MODEL_ORDER = ["2B", "4B", "31B"]
DECODING_ORDER = ["Stochastic", "Greedy"]

VOTE_FILE_REL = Path("base") / "voting" / "prompt_v4" / "vote_stability" / "tables" / "llm_vote_file_level.csv"
VOTE_GAME_REL = Path("base") / "voting" / "prompt_v4" / "vote_stability" / "tables" / "llm_vote_game_level.csv"

CROSS_MODEL_FEATURES_CSV = ANALYSIS_ROOT / "cross_model" / "voting" / "prompt_v4" / "tables" / "cross_model_game_features.csv"
VILLAGE_DISPERSION_CSV = ANALYSIS_ROOT / "human_outcomes" / "prompt_v4" / "tables" / "village_vote_dispersion.csv"
ACC_TARGETS_ROOT = DATA_PROCESSED / "accusation_transcripts" / "acc_targets"
IC_FEATURES_CSV = DATA_PROCESSED / "identity_claim_transcripts" / "ic_targets" / "player_conflict_features.csv"
DIMLEX_ASSIGNMENTS_CSV = REPO_ROOT / "src" / "justification_analysis" / "dimlex_marker_assignments.csv"

OUTPUT_DIR = ANALYSIS_ROOT / "cross_model" / "voting" / "prompt_v4" / "justification_analysis" / "tables"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_BOOT, ALPHA = 5000, 0.05
RNG_BOOT = np.random.default_rng(0)

for name, path in [
    ("CROSS_MODEL_FEATURES_CSV", CROSS_MODEL_FEATURES_CSV),
    ("VILLAGE_DISPERSION_CSV", VILLAGE_DISPERSION_CSV),
    ("IC_FEATURES_CSV", IC_FEATURES_CSV),
    ("DIMLEX_ASSIGNMENTS_CSV", DIMLEX_ASSIGNMENTS_CSV),
    ("ACC_TARGETS_ROOT", ACC_TARGETS_ROOT),
]:
    if not path.exists():
        raise FileNotFoundError(f"{name} not found at {path}")

print("REPO_ROOT:", REPO_ROOT)
print("All configured paths exist.")

REPO_ROOT: C:\Users\annab\Documents\GitHub\masters_thesis_sdg
All configured paths exist.


## Load per-model vote tables, derive base populations

`VOTES_TEXT` keeps every row with real justification text (`status !=
"failed_parse"`) -- used by Analysis 3, which only needs text, not a vote
target. `VOTES_TARGETED` additionally drops `circle_vote` rows
(`status == "player_vote"` only) -- used by Analyses 1, 2 and 4, which all
condition on a real `chosen_player_name`.

In [2]:
def find_model_csv(folder_pattern, relative_path):
    matches = [d / relative_path for d in ANALYSIS_ROOT.glob(folder_pattern) if (d / relative_path).exists()]
    if not matches:
        raise FileNotFoundError(f"No file found for pattern {folder_pattern!r} / {relative_path}")
    if len(matches) > 1:
        raise RuntimeError(f"Multiple matches for {folder_pattern!r}: {matches}")
    return matches[0]


file_frames, game_frames = [], []
for model_name, pattern in MODEL_FOLDER_PATTERNS.items():
    f = pd.read_csv(find_model_csv(pattern, VOTE_FILE_REL))
    f["model"] = model_name
    file_frames.append(f)
    g = pd.read_csv(find_model_csv(pattern, VOTE_GAME_REL))
    g["model"] = model_name
    game_frames.append(g)

votes_raw = pd.concat(file_frames, ignore_index=True)
games_raw = pd.concat(game_frames, ignore_index=True)
print("votes_raw:", votes_raw.shape, " games_raw:", games_raw.shape)

votes_raw["decoding_group"] = np.where(votes_raw["decoding"].str.lower().eq("stochastic"), "Stochastic", "Greedy")
votes_raw["justification"] = votes_raw["justification"].fillna("")
votes_raw["n_words"] = votes_raw["justification"].str.split().str.len().clip(lower=1)
votes_raw["justification_id"] = np.arange(len(votes_raw))

status_counts = votes_raw.groupby(["model", "decoding_group", "status"]).size().unstack(fill_value=0)
print("\nRow counts by status (excluded from VOTES_TEXT/VOTES_TARGETED as noted):")
print(status_counts)

VOTES_TEXT = votes_raw[votes_raw["status"] != "failed_parse"].copy()
VOTES_TARGETED = votes_raw[votes_raw["status"] == "player_vote"].copy()
print(f"\nVOTES_TEXT: {len(VOTES_TEXT)} rows (drops failed_parse only)")
print(f"VOTES_TARGETED: {len(VOTES_TARGETED)} rows (drops failed_parse + circle_vote)")

# SELECTION DIAGNOSTIC: circle votes ("No Werewolf") have no named target, so every
# target-conditioned analysis (1, 2, 4, 5, 6, 7) silently drops them. The models abstain
# at very different rates, so those analyses cover different FRACTIONS of each model's
# behaviour -- this must be disclosed, not assumed away.
_st = votes_raw.groupby(["model", "status"]).size().unstack(fill_value=0)
for c in ("circle_vote", "failed_parse", "player_vote"):
    if c not in _st.columns:
        _st[c] = 0
_st["total"] = _st.sum(axis=1)
_st["pct_circle_vote"] = (100 * _st["circle_vote"] / _st["total"]).round(1)
_st["pct_covered_by_targeted_analyses"] = (100 * _st["player_vote"] / _st["total"]).round(1)
circle_vote_selection = _st.reset_index()
print("\nSELECTION DIAGNOSTIC -- abstention ('No Werewolf') rate differs sharply by model:")
print(circle_vote_selection.to_string(index=False))
print("\n-> target-conditioned analyses cover "
      f"{circle_vote_selection['pct_covered_by_targeted_analyses'].min():.0f}%-"
      f"{circle_vote_selection['pct_covered_by_targeted_analyses'].max():.0f}% of a model's outputs "
      "depending on the model.\n   The excluded votes are plausibly the least-confident ones, so the "
      "analysed subset may\n   flatter the most-abstaining model. Cross-model comparisons are over "
      "unequal slices of behaviour.")

votes_raw: (2292, 16)  games_raw: (573, 22)



Row counts by status (excluded from VOTES_TEXT/VOTES_TARGETED as noted):
status                circle_vote  failed_parse  player_vote
model decoding_group                                        
2B    Greedy                   48             0          143
      Stochastic              131             5          437
31B   Greedy                   24             0          167
      Stochastic               69             0          504
4B    Greedy                   18             0          173
      Stochastic               34             0          539

VOTES_TEXT: 2287 rows (drops failed_parse only)
VOTES_TARGETED: 1963 rows (drops failed_parse + circle_vote)

SELECTION DIAGNOSTIC -- abstention ('No Werewolf') rate differs sharply by model:
model  circle_vote  failed_parse  player_vote  total  pct_circle_vote  pct_covered_by_targeted_analyses
   2B          179             5          580    764             23.4                              75.9
  31B           93             0     

## Canonical keys for the human-side annotation files

`game_id` strings are byte-identical across `llm_vote_file_level.csv`,
`llm_vote_game_level.csv`, `cross_model_game_features.csv` and
`village_vote_dispersion.csv`, so those join directly on `game_id`. The
accusation-target JSONs and `player_conflict_features.csv` key on their own
`source`/`session`/`game` columns, spelled inconsistently (`#` vs space,
case), so they need the canonical key from notebook 3
(`3_vote_surrogate_model.ipynb`), reused verbatim.

In [3]:
def canonical_session(x):
    x = str(x).strip().replace("#", " ")
    return re.sub(r"\s+", " ", x).lower()


def canonical_game(x):
    m = re.search(r"\d+", str(x))
    return f"game{int(m.group())}" if m else str(x).strip().lower()


def ckey(source, session, game):
    return (str(source).strip(), canonical_session(session), canonical_game(game))

## Marker-matching machinery (shared by DiMLex and the new rule-reference lexicon)

Ported from the main pipeline of `1_dimlex_justification_analysis.ipynb`
(`make_phrase_pattern` + longest-match-wins overlap suppression), generalized
into a small reusable `build_lexicon` / `find_matches` / `rate_table` triplet
so the same machinery serves Analyses 1/2 (DiMLex categories) and Analysis 3
(the new rule-reference lexicon).

In [4]:
WORD_PATTERN = re.compile(r"\b[\w]+(?:['’\-][\w]+)*\b", flags=re.UNICODE)


def make_phrase_pattern(marker):
    tokens = marker.split()
    body = r"\s+".join(re.escape(token) for token in tokens)
    return re.compile(rf"(?<!\w){body}(?!\w)", flags=re.IGNORECASE)


def build_lexicon(entries):
    """entries: iterable of (phrase, category) pairs -> lexicon df sorted longest-marker-first."""
    df = pd.DataFrame(list(entries), columns=["marker", "category"])
    df["pattern"] = df["marker"].map(make_phrase_pattern)
    df["marker_length"] = df["marker"].str.len()
    return df.sort_values(["marker_length", "marker"], ascending=[False, True]).reset_index(drop=True)


def find_matches(text, lexicon_df):
    candidates = []
    for row in lexicon_df.itertuples(index=False):
        for m in row.pattern.finditer(text):
            candidates.append({"category": row.category, "start": m.start(), "end": m.end()})
    candidates.sort(key=lambda c: (-(c["end"] - c["start"]), c["start"]))
    accepted, occupied = [], []
    for c in candidates:
        if any(c["start"] < e and c["end"] > s for s, e in occupied):
            continue
        accepted.append(c)
        occupied.append((c["start"], c["end"]))
    return accepted


def rate_table(df, lexicon_df, text_col="justification"):
    """Returns df.copy() with n_<category> and rate_<category> (per 100 words) columns added."""
    cats = sorted(lexicon_df["category"].unique())
    counts = {c: [] for c in cats}
    for text in df[text_col]:
        tally = {c: 0 for c in cats}
        for m in find_matches(text, lexicon_df):
            tally[m["category"]] += 1
        for c in cats:
            counts[c].append(tally[c])
    out = df.copy()
    for c in cats:
        out[f"n_{c}"] = counts[c]
        out[f"rate_{c}"] = 100 * out[f"n_{c}"] / out["n_words"]
    return out


dimlex_df = pd.read_csv(DIMLEX_ASSIGNMENTS_CSV, encoding="utf-8-sig")
dimlex_entries = list(
    dimlex_df.loc[dimlex_df["use_for_matching"], ["marker", "assigned_category"]].itertuples(index=False, name=None)
)
DIMLEX_LEXICON = build_lexicon(dimlex_entries)
print(f"DiMLex matching lexicon: {len(DIMLEX_LEXICON)} markers across categories {sorted(DIMLEX_LEXICON['category'].unique())}")

V_DIMLEX = rate_table(VOTES_TARGETED, DIMLEX_LEXICON)
# "as" sensitivity check for Temporal, mirrored from the established DiMLex robustness check
_as_count = V_DIMLEX["justification"].str.lower().str.count(r"\bas\b")
V_DIMLEX["rate_Temporal_noas"] = 100 * (V_DIMLEX["n_Temporal"] - _as_count).clip(lower=0) / V_DIMLEX["n_words"]
print(f"V_DIMLEX (VOTES_TARGETED with DiMLex rates): {V_DIMLEX.shape}")

DiMLex matching lexicon: 136 markers across categories ['Comparison', 'Contingency', 'Expansion', 'Temporal']


V_DIMLEX (VOTES_TARGETED with DiMLex rates): (1963, 28)


## Aggregation and bootstrap helpers

`game_then_mean` implements the equal-game-weighting convention. `boot_diff`
is the unpaired percentile-bootstrap contrast (two different, only
partially-overlapping instance subsets, e.g. predictable vs. not). `boot_diff_paired`
is used only in Analysis 3, where all three models share the literal same 191
games, so pairing removes per-transcript confounds. `boot_mean` is a
single-arm CI for Analysis 4's mention proportions.

In [5]:
def game_then_mean(df, cols, group_keys=("model", "decoding_group")):
    keys = [*group_keys, "game_id"]
    return df.groupby(keys)[list(cols)].mean().groupby(list(group_keys)).mean()


def boot_diff(a, b, n_boot=N_BOOT, alpha=ALPHA, rng=RNG_BOOT):
    """Unpaired 95% percentile CI for mean(a) - mean(b); a, b are per-game values, independently resampled."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    if len(a) < 2 or len(b) < 2:
        return np.nan, np.nan, np.nan
    da = a[rng.integers(0, len(a), (n_boot, len(a)))].mean(1)
    db = b[rng.integers(0, len(b), (n_boot, len(b)))].mean(1)
    d = da - db
    lo, hi = np.percentile(d, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(a.mean() - b.mean()), float(lo), float(hi)


def boot_diff_paired(a, b, n_boot=N_BOOT, alpha=ALPHA, rng=RNG_BOOT):
    """Paired 95% percentile CI for mean(a) - mean(b); a, b are per-game arrays aligned on the
    SAME game order (index i refers to the same game in both). Used only for Analysis 3's
    cross-model comparison, where every model is scored on the identical 191-game set."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    assert len(a) == len(b), "paired bootstrap requires aligned arrays"
    n = len(a)
    if n < 2:
        return np.nan, np.nan, np.nan
    idx = rng.integers(0, n, (n_boot, n))
    d = a[idx].mean(1) - b[idx].mean(1)
    lo, hi = np.percentile(d, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(a.mean() - b.mean()), float(lo), float(hi)


def boot_mean(a, n_boot=N_BOOT, alpha=ALPHA, rng=RNG_BOOT):
    """95% percentile CI for a single per-game-weighted mean (Analysis 4 proportions)."""
    a = np.asarray(a, float)
    if len(a) < 2:
        return (float(a.mean()) if len(a) else np.nan), np.nan, np.nan
    boots = a[rng.integers(0, len(a), (n_boot, len(a)))].mean(1)
    lo, hi = np.percentile(boots, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(a.mean()), float(lo), float(hi)


def contrast_table(df, value_col, split_col, label_true, label_false, group_keys=("model", "decoding_group")):
    """Unpaired two-group contrast: per-game mean first within each split, per-game-weighted,
    reported with n_games and n_instances for both arms."""
    per_game = (
        df.groupby([*group_keys, split_col, "game_id"])[value_col]
        .agg(value="mean", n_instances="size")
        .reset_index()
    )
    rows = []
    for gkey, grp in per_game.groupby(list(group_keys)):
        a = grp.loc[grp[split_col] == True, "value"].values
        b = grp.loc[grp[split_col] == False, "value"].values
        n_inst_a = int(grp.loc[grp[split_col] == True, "n_instances"].sum())
        n_inst_b = int(grp.loc[grp[split_col] == False, "n_instances"].sum())
        diff, lo, hi = boot_diff(a, b)
        gkey_tuple = gkey if isinstance(gkey, tuple) else (gkey,)
        rows.append({
            **dict(zip(group_keys, gkey_tuple)),
            f"n_games_{label_true}": len(a),
            f"n_games_{label_false}": len(b),
            f"n_instances_{label_true}": n_inst_a,
            f"n_instances_{label_false}": n_inst_b,
            f"mean_{label_true}": round(float(a.mean()), 3) if len(a) else np.nan,
            f"mean_{label_false}": round(float(b.mean()), 3) if len(b) else np.nan,
            "diff": round(diff, 3) if not np.isnan(diff) else np.nan,
            "ci_low": round(lo, 3) if not np.isnan(lo) else np.nan,
            "ci_high": round(hi, 3) if not np.isnan(hi) else np.nan,
            "excludes_zero": bool(not np.isnan(lo) and (lo > 0 or hi < 0)),
            "inconclusive_at_this_n": bool(np.isnan(lo) or not (lo > 0 or hi < 0)),
        })
    return pd.DataFrame(rows)


print("Every contrast below reports n_games and n_instances for both arms; any CI spanning 0")
print("is flagged inconclusive_at_this_n and is not narrated as a real difference in the text.")

Every contrast below reports n_games and n_instances for both arms; any CI spanning 0
is flagged inconclusive_at_this_n and is not narrated as a real difference in the text.


### Probability-weighted contrast (companion to `contrast_table`)

`contrast_table` treats class membership as binary presence: a game with a
2-1 split across its 3 stochastic runs counts with FULL weight in BOTH arms
of a contrast, identical to a game that is a clean 3-of-3. That throws away
exactly the information a T=1-sampling design is supposed to carry -- the
split ratio itself. `weighted_contrast_table` instead weights each game's
contribution to a class by that class's *share* of the game's total valid
instances (a 2-1 split contributes 0.67 to one arm and 0.33 to the other,
not 1.0 to both), consistent with how `cross_model_game_features.csv`'s own
`werewolf`/`human_modal`/`other_human_target` columns are already
vote-share-weighted rather than hard-classified. Both arms are resampled
from the same game pool per bootstrap draw (paired), since they are
fractional shares of the same underlying games, not independent subsets.
Used as a labeled companion alongside `contrast_table` wherever this
notebook splits votes into herded/independent -- the hard-classification
numbers already reported are left untouched.

In [6]:
def boot_diff_weighted(value_a, weight_a, value_b, weight_b, n_boot=N_BOOT, alpha=ALPHA, rng=RNG_BOOT):
    """Weighted-mean-difference 95% percentile CI. value_a/weight_a and value_b/weight_b are
    parallel per-game arrays (a game not contributing to an arm has weight 0 there). Both
    arms are resampled from the SAME game index per draw (paired) since they are fractional
    shares of one game pool, not independent samples."""
    value_a, weight_a, value_b, weight_b = (np.asarray(x, float) for x in (value_a, weight_a, value_b, weight_b))
    n = len(value_a)
    if n < 2 or weight_a.sum() == 0 or weight_b.sum() == 0:
        return np.nan, np.nan, np.nan
    idx = rng.integers(0, n, (n_boot, n))
    wa, va = weight_a[idx], value_a[idx]
    wb, vb = weight_b[idx], value_b[idx]
    sum_wa, sum_wb = wa.sum(1), wb.sum(1)
    mean_a = np.divide((wa * va).sum(1), sum_wa, out=np.full(n_boot, np.nan), where=sum_wa > 0)
    mean_b = np.divide((wb * vb).sum(1), sum_wb, out=np.full(n_boot, np.nan), where=sum_wb > 0)
    d = (mean_a - mean_b)
    d = d[~np.isnan(d)]
    if len(d) < 0.5 * n_boot:
        return np.nan, np.nan, np.nan
    lo, hi = np.percentile(d, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    observed = (weight_a * value_a).sum() / weight_a.sum() - (weight_b * value_b).sum() / weight_b.sum()
    return float(observed), float(lo), float(hi)


def weighted_contrast_table(df, value_col, class_col, label_true, label_false, group_keys=("model", "decoding_group")):
    """df must include ALL classes for the class_col (e.g. 'other' too) so each game's
    per-class weight is a true share of its total valid instances, not just of the two
    arms being contrasted."""
    rows = []
    for gkey, g in df.groupby(list(group_keys)):
        per_game_class = (
            g.groupby(["game_id", class_col])[value_col].agg(n="size", val="mean").reset_index()
        )
        n_total = g.groupby("game_id").size().rename("n_total").reset_index()
        per_game_class = per_game_class.merge(n_total, on="game_id")
        per_game_class["weight"] = per_game_class["n"] / per_game_class["n_total"]
        piv_w = per_game_class.pivot(index="game_id", columns=class_col, values="weight").reindex(columns=[label_true, label_false]).fillna(0.0)
        piv_v = per_game_class.pivot(index="game_id", columns=class_col, values="val").reindex(columns=[label_true, label_false]).fillna(0.0)
        weight_true, weight_false = piv_w[label_true].values, piv_w[label_false].values
        value_true, value_false = piv_v[label_true].values, piv_v[label_false].values
        diff, lo, hi = boot_diff_weighted(value_true, weight_true, value_false, weight_false)
        gkey_tuple = gkey if isinstance(gkey, tuple) else (gkey,)
        rows.append({
            **dict(zip(group_keys, gkey_tuple)),
            "n_games": len(piv_w),
            f"weighted_mean_{label_true}": round(float((weight_true * value_true).sum() / weight_true.sum()), 3) if weight_true.sum() > 0 else np.nan,
            f"weighted_mean_{label_false}": round(float((weight_false * value_false).sum() / weight_false.sum()), 3) if weight_false.sum() > 0 else np.nan,
            "diff": round(diff, 3) if not np.isnan(diff) else np.nan,
            "ci_low": round(lo, 3) if not np.isnan(lo) else np.nan,
            "ci_high": round(hi, 3) if not np.isnan(hi) else np.nan,
            "excludes_zero": bool(not np.isnan(lo) and (lo > 0 or hi < 0)),
            "inconclusive_at_this_n": bool(np.isnan(lo) or not (lo > 0 or hi < 0)),
        })
    return pd.DataFrame(rows)

# Analysis 1 -- Markers x surrogate-predictability (proxy)

> **Serves RQ1 + RQ2.** If a model reasons its way to a vote, its "reasoning language"
> should be denser on votes that social salience does *not* explain. If the density is
> flat, reasoning language is decoration rather than a trace of reasoning.
> **Result: flat in 4 of 6 cells -- decoration.** This null is what motivates abandoning
> language-based measures for the ground-truth checks in Analyses 5-7.

The surrogate model (`3_vote_surrogate_model.ipynb`) does not save per-instance
out-of-fold predictions, so this uses the necessary **proxy**: a vote is
"predictable" if its target is that game's most-accused player
(werewolf-type accusations only, matching the surrogate's actual feature --
deception-type accusations exist in the same data but are a separate,
unused feature there and are left out here too) **or** a Werewolf
self-claimer. This is not the surrogate's held-out score; it is a
necessity-based proxy built from the same two dialogue features. If
Contingency density is flat between predictable and unpredictable votes, that
is evidence reasoning language is decorative rather than tracking the actual
evidence driving the vote.

In [7]:
acc_rows = []
for p in sorted(ACC_TARGETS_ROOT.rglob("*.json")):
    try:
        rec = json.loads(p.read_text(encoding="utf-8"))
    except (json.JSONDecodeError, OSError):
        continue
    meta = rec.get("metadata", {})
    if not all(meta.get(k) for k in ("source", "session", "game")):
        continue
    key = ckey(meta["source"], meta["session"], meta["game"])
    for item in rec.get("items", []):
        for rel in item.get("relations", []):
            if rel.get("type") != "werewolf":
                continue
            for pl in rel.get("accused", []):
                if pl != "UNKNOWN":
                    acc_rows.append({"key": key, "player": pl})

acc_df = pd.DataFrame(acc_rows)
werewolf_counts = (
    acc_df.groupby(["key", "player"]).size().rename("werewolf_count").reset_index()
    if not acc_df.empty else pd.DataFrame(columns=["key", "player", "werewolf_count"])
)
print(f"werewolf-type accusation events: {len(acc_df)}; games covered: {werewolf_counts['key'].nunique()}")

ic = pd.read_csv(IC_FEATURES_CSV)
ic["key"] = [ckey(s, se, g) for s, se, g in zip(ic["source"], ic["session"], ic["game"])]


def parse_roles(v):
    try:
        return set(json.loads(v)) if isinstance(v, str) else set()
    except json.JSONDecodeError:
        return set()


ic["claims_werewolf"] = ic["roles_claimed"].apply(lambda v: int("Werewolf" in parse_roles(v)))
ic["roles_set"] = ic["roles_claimed"].apply(parse_roles)
ic["is_self_contradiction_bool"] = ic["is_self_contradiction"].astype(str).str.lower().eq("true")
ic_feats = ic[["key", "player", "claims_werewolf", "roles_set", "is_self_contradiction_bool"]]
print(f"identity-claim rows: {len(ic_feats)}; games covered: {ic_feats['key'].nunique()}")
print(f"rows flagged is_self_contradiction: {int(ic_feats['is_self_contradiction_bool'].sum())}")

werewolf-type accusation events: 1184; games covered: 173
identity-claim rows: 632; games covered: 189
rows flagged is_self_contradiction: 142


In [8]:
def name_map(players):
    return {p.strip().lower(): p for p in players}


roster_by_game = (
    games_raw.drop_duplicates(subset="game_id")
    .assign(players=lambda d: d["player_names"].map(json.loads),
            source=lambda d: d["source"], session_name=lambda d: d["session_name"], game_key=lambda d: d["game_key"])
)

candidate_rows = []
for row in roster_by_game.itertuples(index=False):
    game_id, players = row.game_id, row.players
    key = ckey(row.source, row.session_name, row.game_key)
    nmap = name_map(players)
    wc = {p: 0 for p in players}
    for r in werewolf_counts[werewolf_counts["key"] == key].itertuples(index=False):
        canon = nmap.get(str(r.player).strip().lower())
        if canon is not None:
            wc[canon] += int(r.werewolf_count)
    cw = {p: 0 for p in players}
    rs = {p: frozenset() for p in players}
    sc = {p: False for p in players}
    for r in ic_feats[ic_feats["key"] == key].itertuples(index=False):
        canon = nmap.get(str(r.player).strip().lower())
        if canon is not None:
            cw[canon] = max(cw[canon], int(r.claims_werewolf))
            rs[canon] = frozenset(r.roles_set)
            sc[canon] = bool(r.is_self_contradiction_bool)
    for p in players:
        candidate_rows.append({
            "game_id": game_id, "player": p, "werewolf_count": wc[p], "claims_werewolf": cw[p],
            "roles_claimed_set": rs[p], "is_self_contradiction": sc[p],
        })

candidates = pd.DataFrame(candidate_rows)
print(f"candidate rows: {len(candidates)} over {candidates['game_id'].nunique()} games")


def most_accused(group):
    if group["werewolf_count"].max() == 0:
        return None  # undefined: no recorded accusation evidence in this game at all
    top = group["werewolf_count"].max()
    return sorted(group.loc[group["werewolf_count"] == top, "player"])[0]  # deterministic alphabetical tie-break


most_accused_by_game = candidates.groupby("game_id").apply(most_accused, include_groups=False).to_dict()
claims_werewolf_by_game = (
    candidates.groupby("game_id")
    .apply(lambda g: set(g.loc[g["claims_werewolf"] == 1, "player"]), include_groups=False)
    .to_dict()
)

n_defined = sum(1 for v in most_accused_by_game.values() if v is not None)
n_with_claim = sum(1 for v in claims_werewolf_by_game.values() if len(v) > 0)
print(f"games with most_accused_player defined: {n_defined} / {len(most_accused_by_game)}")
print(f"games with all-zero accusations (most_accused_player = None): {len(most_accused_by_game) - n_defined}")
print(f"games with >=1 Werewolf self-claimer: {n_with_claim} / {len(claims_werewolf_by_game)}")

# Used by Analysis 6 (ground-truth claim-contradiction flagging)
claimed_roles_by_game_player = candidates.set_index(["game_id", "player"])["roles_claimed_set"].to_dict()
self_contradiction_by_game = (
    candidates.groupby("game_id")
    .apply(lambda g: set(g.loc[g["is_self_contradiction"], "player"]), include_groups=False)
    .to_dict()
)
n_games_with_contradiction = sum(1 for v in self_contradiction_by_game.values() if len(v) > 0)
print(f"games with >=1 self-contradicting claimant: {n_games_with_contradiction} / {len(self_contradiction_by_game)}")

candidate rows: 864 over 191 games


games with most_accused_player defined: 173 / 191
games with all-zero accusations (most_accused_player = None): 18
games with >=1 Werewolf self-claimer: 76 / 191
games with >=1 self-contradicting claimant: 95 / 191


### Feature-availability diagnostic

The predictability proxy has two pillars, and **not every game contains both** -- many
games have no player who ever literally claims "I am the Werewolf". If a game contains
*neither* pillar, the proxy cannot fire there for structural reasons, and any vote in that
game is scored "unpredictable" regardless of what the model did. That would contaminate
the unpredictable arm with games where there was simply no salience signal to follow.

The two pillars are combined with **OR**, so they cover for each other; the question is how
many games have neither. Note also that a game with no self-claimer is a genuine **zero**
("nobody claimed Werewolf"), not missing data -- every roster player is initialised to 0
before annotated values are overlaid, exactly as in the surrogate notebook.

In [9]:
avail_rows = []
for game_id in sorted(roster_by_game["game_id"]):
    has_acc = most_accused_by_game.get(game_id) is not None
    has_claim = len(claims_werewolf_by_game.get(game_id, set())) > 0
    avail_rows.append({"game_id": game_id, "has_accusation_pillar": has_acc,
                       "has_self_claim_pillar": has_claim,
                       "has_any_pillar": has_acc or has_claim})
feature_availability = pd.DataFrame(avail_rows)
n = len(feature_availability)
feature_availability_summary = pd.DataFrame([{
    "n_games": n,
    "n_with_accusation_pillar": int(feature_availability["has_accusation_pillar"].sum()),
    "n_with_self_claim_pillar": int(feature_availability["has_self_claim_pillar"].sum()),
    "n_with_any_pillar": int(feature_availability["has_any_pillar"].sum()),
    "n_with_neither_pillar": int((~feature_availability["has_any_pillar"]).sum()),
    "pct_with_any_pillar": round(100 * feature_availability["has_any_pillar"].mean(), 1),
}])
print(feature_availability_summary.to_string(index=False))
print(f"\n-> the proxy is structurally unable to fire in "
      f"{int((~feature_availability['has_any_pillar']).sum())} of {n} games "
      f"({100*(~feature_availability['has_any_pillar']).mean():.1f}%); those votes are "
      f"scored unpredictable for lack of signal, not for model behaviour.")

 n_games  n_with_accusation_pillar  n_with_self_claim_pillar  n_with_any_pillar  n_with_neither_pillar  pct_with_any_pillar
     191                       173                        76                176                     15                 92.1

-> the proxy is structurally unable to fire in 15 of 191 games (7.9%); those votes are scored unpredictable for lack of signal, not for model behaviour.


In [10]:
V1 = V_DIMLEX.copy()
V1["most_accused_player"] = V1["game_id"].map(most_accused_by_game)
V1["is_predictable"] = (
    (V1["chosen_player_name"] == V1["most_accused_player"])
    | V1.apply(lambda r: r["chosen_player_name"] in claims_werewolf_by_game.get(r["game_id"], set()), axis=1)
)

print(V1.groupby(["model", "decoding_group", "is_predictable"]).size().unstack(fill_value=0))

surrogate_means = (
    V1.groupby(["model", "decoding_group", "is_predictable", "game_id"])["rate_Contingency"]
    .mean().reset_index()
    .groupby(["model", "decoding_group", "is_predictable"])["rate_Contingency"].mean().unstack()
    .rename(columns={True: "mean_predictable", False: "mean_unpredictable"})
    .reset_index()
)
surrogate_contingency_ci = contrast_table(V1, "rate_Contingency", "is_predictable", "predictable", "unpredictable")
print("\nContingency density: predictable vs. unpredictable votes")
print(surrogate_contingency_ci)

is_predictable        False  True 
model decoding_group              
2B    Greedy             49     94
      Stochastic        153    284
31B   Greedy             77     90
      Stochastic        209    295
4B    Greedy             62    111
      Stochastic        214    325



Contingency density: predictable vs. unpredictable votes
  model decoding_group  n_games_predictable  n_games_unpredictable  n_instances_predictable  n_instances_unpredictable  mean_predictable  \
0    2B         Greedy                   94                     49                       94                         49             3.293   
1    2B     Stochastic                  126                     83                      284                        153             3.135   
2   31B         Greedy                   90                     77                       90                         77             2.435   
3   31B     Stochastic                  121                     94                      295                        209             2.689   
4    4B         Greedy                  111                     62                      111                         62             2.326   
5    4B     Stochastic                  134                     99                      325           

### Appendix -- patch to save true out-of-fold predictions (not implemented here)

In `notebooks/03_llm_voting_outcome_analysis/3_vote_surrogate_model.ipynb`'s
`cv_evaluate`, inside the `for fold in range(N_FOLDS):` loop, after
`te["score"] = model.predict_proba(X_te)[:, 1]`, accumulate
`te[["key", "instance", "player", "label", "score"]].assign(method=method, seed=seed, fold=fold)`
into a list defined before the seed loop. After both loops finish,
`pd.concat(oof_rows, ignore_index=True)` and write it to
`OUTPUT_DIR / "surrogate_oof_predictions.csv"`. `train_df["instance"]` is a
`(key, run_label)` tuple, so the true out-of-fold grain would be
`(game key, run_label, player)` -- one row per candidate per fold per seed
per method. A future Analysis 1 could then replace the proxy `is_predictable`
above with the actual OOF top-1 pick (`score == score.max()` within
`instance`), averaged across the 3 CV seeds, instead of the accusation/claim
proxy used here.

# Analysis 2 -- Markers x game class (herded vs. independent)

> **Serves RQ2.** Restricted to *dissociated* games -- the ones where the human crowd's
> most-popular vote was **wrong**. Does a model that resists the crowd (votes the true
> Werewolf) write differently from one that follows it into the error?
> **Result: essentially flat (1 of 12 cells clears the CI).** Crowd-following is real in
> the *votes* (see the 2x vote-mass finding) but leaves no visible trace in the *text*.

Restricted to `dissociated` games (the human crowd's modal target is
disjoint from the true Werewolf set -- reusing the exact `game_class`
already computed in `2_cross_model_vote_comparison.ipynb`, not recomputed
here). Within a dissociated game, a vote instance is **independent** if its
target is the true Werewolf, **herded** if its target is the crowd's wrong
modal pick, or **other** (targeted a real player who is neither -- excluded
from the headline contrast, counted separately). By construction of
`dissociated`, herded and independent never overlap for a single instance.

In [11]:
cmf = pd.read_csv(CROSS_MODEL_FEATURES_CSV)
dissociated_games = cmf.loc[cmf["game_class"] == "dissociated", ["model", "game_id"]].drop_duplicates()
print(f"dissociated (model, game_id) pairs: {len(dissociated_games)}")

village = pd.read_csv(VILLAGE_DISPERSION_CSV)
village["village_top_target_names"] = village["village_top_target_names"].apply(json.loads)
village_targets_by_game = village.set_index("game_id")["village_top_target_names"].apply(set).to_dict()

wolves_by_model_game = (
    games_raw.assign(wolves=lambda d: d["correct_player_names"].map(lambda v: set(json.loads(v))))
    .set_index(["model", "game_id"])["wolves"].to_dict()
)

V2 = (
    V_DIMLEX.merge(dissociated_games, on=["model", "game_id"], how="inner")
)


def classify_instance(row):
    wolves = wolves_by_model_game.get((row["model"], row["game_id"]), set())
    village_top = village_targets_by_game.get(row["game_id"], set())
    if row["chosen_player_name"] in wolves:
        return "independent"
    if row["chosen_player_name"] in village_top:
        return "herded"
    return "other"


V2["vote_class"] = V2.apply(classify_instance, axis=1)
vote_class_counts = V2.groupby(["model", "decoding_group", "vote_class"]).size().unstack(fill_value=0)
print(vote_class_counts)

excluded_other = V2[V2["vote_class"] == "other"].groupby(["model", "decoding_group"]).size().rename("n_excluded_other").reset_index()

V2_contrast = V2[V2["vote_class"] != "other"].copy()
V2_contrast["is_herded"] = V2_contrast["vote_class"].eq("herded")

herded_contingency_ci = contrast_table(V2_contrast, "rate_Contingency", "is_herded", "herded", "independent")
herded_contingency_ci.insert(2, "marker_category", "Contingency")
herded_temporal_ci = contrast_table(V2_contrast, "rate_Temporal", "is_herded", "herded", "independent")
herded_temporal_ci.insert(2, "marker_category", "Temporal")
markers_by_herded_independent_ci = pd.concat([herded_contingency_ci, herded_temporal_ci], ignore_index=True)
print("\nContingency/Temporal density: herded vs. independent votes (dissociated games only)")
print(markers_by_herded_independent_ci)

dissociated (model, game_id) pairs: 141


vote_class            herded  independent  other
model decoding_group                            
2B    Greedy              17           12     11
      Stochastic          57           35     24
31B   Greedy              19           12     11
      Stochastic          67           26     36
4B    Greedy              19           11     15
      Stochastic          58           36     44



Contingency/Temporal density: herded vs. independent votes (dissociated games only)
   model decoding_group marker_category  n_games_herded  n_games_independent  n_instances_herded  n_instances_independent  mean_herded  \
0     2B         Greedy     Contingency              17                   12                  17                       12        3.399   
1     2B     Stochastic     Contingency              28                   19                  57                       35        3.042   
2    31B         Greedy     Contingency              19                   12                  19                       12        2.430   
3    31B     Stochastic     Contingency              29                   15                  67                       26        2.736   
4     4B         Greedy     Contingency              19                   11                  19                       11        2.063   
5     4B     Stochastic     Contingency              25                   17           

### Probability-weighted companion

Same contrast, but a game's vote split across its (up to 3) stochastic runs
is weighted by its actual share rather than hard binary presence -- see the
"Probability-weighted contrast" note in the Aggregation section. Computed on
`V2` (all three vote classes still present, so weights are true shares of
each game's total valid instances, `other` included).

In [12]:
herded_contingency_weighted_ci = weighted_contrast_table(V2, "rate_Contingency", "vote_class", "herded", "independent")
herded_contingency_weighted_ci.insert(2, "marker_category", "Contingency")
herded_temporal_weighted_ci = weighted_contrast_table(V2, "rate_Temporal", "vote_class", "herded", "independent")
herded_temporal_weighted_ci.insert(2, "marker_category", "Temporal")
markers_by_herded_independent_weighted_ci = pd.concat([herded_contingency_weighted_ci, herded_temporal_weighted_ci], ignore_index=True)
print(markers_by_herded_independent_weighted_ci)

   model decoding_group marker_category  n_games  weighted_mean_herded  weighted_mean_independent   diff  ci_low  ci_high  excludes_zero  \
0     2B         Greedy     Contingency       40                 3.399                      2.515  0.883  -0.374    2.130          False   
1     2B     Stochastic     Contingency       46                 3.006                      2.976  0.030  -0.822    0.833          False   
2    31B         Greedy     Contingency       42                 2.430                      2.148  0.282  -0.966    1.631          False   
3    31B     Stochastic     Contingency       45                 2.604                      2.883 -0.279  -1.361    0.824          False   
4     4B         Greedy     Contingency       45                 2.063                      3.083 -1.020  -2.208    0.174          False   
5     4B     Stochastic     Contingency       47                 1.977                      2.677 -0.700  -1.456    0.012          False   
6     2B         Gre

# Role vocabulary (shared machinery for Analyses 5-6)

The 13 canonical ONUW role names, derived **dynamically from the dataset** (the union of
`start_roles` / `end_roles` across all 191 games), minus two slot values that are not role
names: `"Moderator"` (a transcript artifact -- the human running the game) and
`"Center Card"` (a real end-state after a blind Drunk swap, but a *location*, not an
identity). This vocabulary is data-derived, not authored, and is the only lexicon used by
the ground-truth analyses below.

### Removed: the analyst-authored rule-reference lexicon

An earlier version of this notebook counted hand-written "rule vocabulary" (`center`,
`swap`, `rob`, `wake up`, `peek`, `night`, ...) and reported per-model densities.
**It has been removed.** The reasons are worth recording, because they motivate the
ground-truth approach used from Analysis 5 onward:

- **It was invented, not grounded.** The terms came from the analyst, not from the rules
  text the models actually received (`src/prompts/onuw_rules_v2.txt`). `rob`/`robbed`
  never appear in those rules, which say the Robber *"swaps their card"*; `peek` does not
  appear either, and `peek`/`peeked`/`peeking` matched **zero** times across all 2,287
  justifications -- vocabulary guessed rather than observed.
- **It measured vocabulary, not correctness.** A model can write "Robber" without tracking
  what the Robber did. Analyses 5-6 test the latter directly.
- **The night-order category was structurally confounded.** It could not distinguish a
  model *relaying another player's public claim* ("Erin said she looked at the centre")
  from a model *using a rule as a premise*, and it was far too sparse to support inference
  (12% of justifications, 5% excluding the polysemous bare "night").
- **It was partly redundant with the task itself.** "Werewolf" is a role name and appears
  in 94-99.5% of all justifications regardless of model, so role-name density is
  substantially a measure of restating the question.

**Future work:** a defensible replacement would derive its vocabulary from
`src/prompts/onuw_rules_v2.txt` -- the exact rules text shown to the models -- rather than
from analyst intuition, and validate it against the ground-truth measures in Analyses 5-6
before being reported. That is left as a deliberate next step, not attempted here.

**Unaffected:** the DiMLex discourse-marker results (Temporal / Contingency, notebook 1).
That lexicon is externally published (PDTB-derived) and independently validated, and its
Temporal category is the one text-based signal that does track the capability ordering.

In [13]:
def role_vocab(games_df):
    roles = set()
    for col in ("start_roles", "end_roles"):
        for v in games_df[col].dropna():
            roles.update(json.loads(v))
    return roles


# "Moderator": transcript-slot artifact (the human running the game), not an ONUW role.
# "Center Card": a real end-state (the Drunk blind-swaps) but a location, not an identity.
NON_ROLE_SLOT_VALUES = {"Moderator", "Center Card"}
ALL_ROLES = role_vocab(games_raw) - NON_ROLE_SLOT_VALUES
print(f"Role-slot values excluded as non-roles: {sorted(NON_ROLE_SLOT_VALUES)}")
print(f"Role vocabulary ({len(ALL_ROLES)} roles, data-derived):")
print(sorted(ALL_ROLES))

Role-slot values excluded as non-roles: ['Center Card', 'Moderator']
Role vocabulary (13 roles, data-derived):
['Doppelganger', 'Drunk', 'Hunter', 'Insomniac', 'Mason', 'Minion', 'Revealer', 'Robber', 'Seer', 'Tanner', 'Troublemaker', 'Villager', 'Werewolf']


# Analysis 4 -- Mention-check (faithfulness proxy)

> **Serves RQ2.** When a vote *is* explained by social salience (the target was the
> most-accused player, or self-claimed Werewolf), does the justification actually name
> that evidence -- or does the model follow the crowd while writing about something else?
> **Result: ~98-100% across all models -- a ceiling effect.** Models are transparent
> about following social salience; they do not conceal it. Little discriminating power
> between models, but it rules out "silent" crowd-following.

**Caveat:** mentioning an entity or Werewolf-claim language is not proof the
justification's reasoning is causally faithful to that evidence -- it is the
cheapest available proxy, reporting only that the concept was named in the
text. Pillar 1 (accusation): among votes whose target is that game's
most-accused player, does the justification name that player? Pillar 2
(self-claim): among votes whose target self-claimed Werewolf, does the
justification mention the word "werewolf" at all? The two pillars are **not
mutually exclusive** (a target can satisfy both) -- each has an independent
denominator and the percentages should not be summed.

In [14]:
def name_mentioned(text, name):
    return re.search(rf"(?<!\w){re.escape(name)}(?!\w)", text, flags=re.IGNORECASE) is not None


WEREWOLF_WORD = re.compile(r"\bwerewolf\b", flags=re.IGNORECASE)


def werewolf_language_mentioned(text):
    return bool(WEREWOLF_WORD.search(text))


V4 = VOTES_TARGETED.copy()
V4["most_accused_player"] = V4["game_id"].map(most_accused_by_game)
V4["target_is_most_accused"] = V4["chosen_player_name"] == V4["most_accused_player"]
V4["target_claims_werewolf"] = V4.apply(
    lambda r: r["chosen_player_name"] in claims_werewolf_by_game.get(r["game_id"], set()), axis=1
)

pillar1 = V4[V4["target_is_most_accused"]].copy()
pillar1["mentions_target"] = pillar1.apply(lambda r: name_mentioned(r["justification"], r["chosen_player_name"]), axis=1)

pillar2 = V4[V4["target_claims_werewolf"]].copy()
pillar2["mentions_werewolf_language"] = pillar2["justification"].map(werewolf_language_mentioned)

print(f"Pillar 1 (accusation) instances: {len(pillar1)}   Pillar 2 (self-claim) instances: {len(pillar2)}")

Pillar 1 (accusation) instances: 857   Pillar 2 (self-claim) instances: 560


In [15]:
def mention_rate_table(df, mention_col, pillar_label):
    per_game = df.groupby(["model", "decoding_group", "game_id"])[mention_col].mean().reset_index()
    rows = []
    for (mdl, dg), grp in per_game.groupby(["model", "decoding_group"]):
        n_inst = int(df[(df["model"] == mdl) & (df["decoding_group"] == dg)].shape[0])
        mean, lo, hi = boot_mean(grp[mention_col].values)
        rows.append({
            "model": mdl, "decoding_group": dg, "pillar": pillar_label,
            "n_games": len(grp), "n_instances": n_inst,
            "pct_mentioning": round(100 * mean, 1) if not np.isnan(mean) else np.nan,
            "ci_low": round(100 * lo, 1) if not np.isnan(lo) else np.nan,
            "ci_high": round(100 * hi, 1) if not np.isnan(hi) else np.nan,
        })
    return pd.DataFrame(rows)


mention_check = pd.concat([
    mention_rate_table(pillar1, "mentions_target", "accusation_pillar"),
    mention_rate_table(pillar2, "mentions_werewolf_language", "self_claim_pillar"),
], ignore_index=True)
print(mention_check)

   model decoding_group             pillar  n_games  n_instances  pct_mentioning  ci_low  ci_high
0     2B         Greedy  accusation_pillar       66           66           100.0   100.0    100.0
1     2B     Stochastic  accusation_pillar       95          199            97.9    94.7    100.0
2    31B         Greedy  accusation_pillar       69           69           100.0   100.0    100.0
3    31B     Stochastic  accusation_pillar       96          225           100.0   100.0    100.0
4     4B         Greedy  accusation_pillar       78           78           100.0   100.0    100.0
5     4B     Stochastic  accusation_pillar      101          220           100.0   100.0    100.0
6     2B         Greedy  self_claim_pillar       45           45           100.0   100.0    100.0
7     2B     Stochastic  self_claim_pillar       57          136           100.0   100.0    100.0
8    31B         Greedy  self_claim_pillar       33           33           100.0   100.0    100.0
9    31B     Stochas

# Analysis 5 -- Ground-truth swap tracking (algorithmic deduction check)

> ## ★ Primary evidence for RQ1
>
> Every analysis above counted *words*. This one checks *correctness*. In ONUW, night
> actions move cards between players, so a player's **starting** role can differ from
> their **final** role -- and tracking that is the single hardest thing the rules
> require. Ground truth for it already exists in the dataset
> (`voted_player_start_role` / `voted_player_end_role`).
>
> **The test:** on votes where the target's role genuinely changed, does the
> justification name **both** the start role and the end role -- i.e. explicitly narrate
> the swap? This is falsifiable, uses only the 13 canonical role names from the rules
> text, and cannot be satisfied by reasoning-flavoured vocabulary.
>
> **Result: 2B 6-8%, 4B ~19%, 31B 42-46%.** All 6 cross-model contrasts clear the CI, in
> both decoding regimes, and survive both the verbosity and chance-adjusted controls.
> This is the cleanest quantitative answer to RQ1 in the thesis.

The RQ needs an *algorithmic* way to establish rule-based deductive
reasoning, not a word list -- a lexicon can only measure vocabulary, never
correctness. This checks a **falsifiable claim** instead: when the voted
player's role actually changed (`voted_player_start_role !=
voted_player_end_role` -- already-existing ground-truth columns, no new
joins), does the justification correctly name the *current* (end) role?
Uses only the closed, externally-sourced 13-role vocabulary from Analysis 3
(`ALL_ROLES`) -- no analyst-invented phrases.

Mentioning **both** the start role and the end role for the target is the
target class: it is an explicit swap narration ("she started as Villager but
ended as Werewolf"), a stronger and more specific signal than matching the
end role alone, which could result from unrelated evidence (e.g. accusation
counts) without any tracked deduction.

**Caveats:** (1) co-occurrence is not proof of assertion -- the same
"cheapest available proxy" caveat as Analysis 4 applies. (2) This only
diagnoses votes where the target actually swapped; it says nothing about
non-swap votes. (3) The `neither` bucket (mentions some other role) is
reported but not interpreted without further qualitative review.

In [16]:
swap_target = VOTES_TARGETED[VOTES_TARGETED["voted_player_start_role"] != VOTES_TARGETED["voted_player_end_role"]].copy()
print(f"Swap-target votes (voted player's role changed): {len(swap_target)} / {len(VOTES_TARGETED)}")
print(swap_target.groupby(["model", "decoding_group"]).size().unstack(fill_value=0))

ROLE_ATTRIBUTION_PATTERN = re.compile(
    r"(?<!\w)(" + "|".join(re.escape(r) for r in ALL_ROLES) + r")(?!\w)", flags=re.IGNORECASE
)


def get_role_mentions(text):
    return {m.group(1).title() for m in ROLE_ATTRIBUTION_PATTERN.finditer(text)}


def chance_multi_match_prob(n_other, k_prime, m):
    """P(a random m-subset of the n_other 'irrelevant' roles has >=2 members among the
    k_prime 'irrelevant' roles this justification actually happens to mention) -- the null
    baseline for 'both'/'flags_multiple_claims' under a pure verbosity/stylistic-trait
    explanation (the model just names roles a lot, unrelated to tracking this specific fact).
    m=2 (Analysis 5, always a pair) reduces to C(k_prime,2)/C(n_other,2); m>2 (Analysis 6,
    a player can claim more than 2 distinct roles) uses the general hypergeometric tail."""
    if m < 2 or n_other < 2:
        return 0.0
    return float(hypergeom.sf(1, n_other, k_prime, m))


def classify_swap_attribution(row):
    mentions = get_role_mentions(row["justification"])
    has_end = row["voted_player_end_role"] in mentions
    has_start = row["voted_player_start_role"] in mentions
    if has_end and has_start:
        return "both"
    if has_end:
        return "end_only"
    if has_start:
        return "start_only"
    if mentions:
        return "neither"
    return "no_mention"


swap_target["attribution"] = swap_target.apply(classify_swap_attribution, axis=1)
attribution_counts = swap_target.groupby(["model", "decoding_group", "attribution"]).size().unstack(fill_value=0)
print("\nRole-attribution classification on swap-target votes:")
print(attribution_counts)

# Confound check: is this just verbosity? Compare avg n_words across models (all targeted votes).
print("\nAvg n_words per justification (verbosity check):")
print(VOTES_TARGETED.groupby("model")["n_words"].mean().round(1))

Swap-target votes (voted player's role changed): 877 / 1963
decoding_group  Greedy  Stochastic
model                             
2B                  67         196
31B                 70         223
4B                  81         240

Role-attribution classification on swap-target votes:
attribution           both  end_only  neither  no_mention  start_only
model decoding_group                                                 
2B    Greedy             5        21       23           1          17
      Stochastic        13        60       59           0          64
31B   Greedy            32        12        5           1          20
      Stochastic        95        38       11           0          79
4B    Greedy            16        18       23           0          24
      Stochastic        47        57       54           2          80

Avg n_words per justification (verbosity check):


model
2B     63.6
31B    74.5
4B     81.8
Name: n_words, dtype: float64


In [17]:
swap_target["is_both"] = (swap_target["attribution"] == "both").astype(float)
per_game_both = (
    swap_target.groupby(["model", "decoding_group", "game_id"])
    .agg(share_both=("is_both", "mean"), n_instances=("is_both", "size"))
    .reset_index()
)
share_both_by_model = (
    per_game_both.groupby(["model", "decoding_group"])
    .agg(n_games=("game_id", "nunique"), n_instances=("n_instances", "sum"), mean_share_both=("share_both", "mean"))
    .reset_index()
)
print(share_both_by_model.round(3))


def unpaired_cross_model_ci(per_game_df, value_col, decoding_group, model_pairs=(("31B", "2B"), ("31B", "4B"), ("4B", "2B"))):
    sub = per_game_df[per_game_df["decoding_group"] == decoding_group]
    rows = []
    for m1, m2 in model_pairs:
        a = sub.loc[sub["model"] == m1, value_col].values
        b = sub.loc[sub["model"] == m2, value_col].values
        diff, lo, hi = boot_diff(a, b)
        rows.append({
            "decoding_group": decoding_group, "model_a": m1, "model_b": m2,
            "n_games_a": len(a), "n_games_b": len(b),
            "mean_a": round(float(a.mean()), 3) if len(a) else np.nan,
            "mean_b": round(float(b.mean()), 3) if len(b) else np.nan,
            "diff": round(diff, 3) if not np.isnan(diff) else np.nan,
            "ci_low": round(lo, 3) if not np.isnan(lo) else np.nan,
            "ci_high": round(hi, 3) if not np.isnan(hi) else np.nan,
            "excludes_zero": bool(not np.isnan(lo) and (lo > 0 or hi < 0)),
            "inconclusive_at_this_n": bool(np.isnan(lo) or not (lo > 0 or hi < 0)),
        })
    return pd.DataFrame(rows)


swap_target_bootstrap_ci = pd.concat(
    [unpaired_cross_model_ci(per_game_both, "share_both", dg) for dg in DECODING_ORDER], ignore_index=True
)
print("\nCross-model contrast on share_both (unpaired -- swap-target subset differs per model):")
print(swap_target_bootstrap_ci)

  model decoding_group  n_games  n_instances  mean_share_both
0    2B         Greedy       67           67            0.075
1    2B     Stochastic       90          196            0.061
2   31B         Greedy       70           70            0.457
3   31B     Stochastic       87          223            0.420
4    4B         Greedy       81           81            0.198
5    4B     Stochastic      103          240            0.194



Cross-model contrast on share_both (unpaired -- swap-target subset differs per model):
  decoding_group model_a model_b  n_games_a  n_games_b  mean_a  mean_b   diff  ci_low  ci_high  excludes_zero  inconclusive_at_this_n
0     Stochastic     31B      2B         87         90   0.420   0.061  0.358   0.258    0.457           True                   False
1     Stochastic     31B      4B         87        103   0.420   0.194  0.225   0.117    0.335           True                   False
2     Stochastic      4B      2B        103         90   0.194   0.061  0.133   0.060    0.209           True                   False
3         Greedy     31B      2B         70         67   0.457   0.075  0.383   0.252    0.512           True                   False
4         Greedy     31B      4B         70         81   0.457   0.198  0.260   0.119    0.407           True                   False
5         Greedy      4B      2B         81         67   0.198   0.075  0.123   0.014    0.229           Tru

### Precision layer -- DiMLex-connective check (reuses existing validated machinery)

Among `both` instances, does a DiMLex Temporal/Comparison marker (from the
already-built `DIMLEX_LEXICON`) sit textually between the start-role mention
and the end-role mention? Evidence the two mentions are linked by an
explicit contrast/temporal connective ("started as X **but** is now Y")
rather than being two disconnected observations in the same paragraph. No
new vocabulary is introduced -- this reuses the same published lexicon
already used in Analyses 1 and 2.

In [18]:
TEMPORAL_COMPARISON_CATS = {"Temporal", "Comparison"}


def connective_between_roles(text, start_role, end_role):
    start_matches = list(make_phrase_pattern(start_role).finditer(text))
    end_matches = list(make_phrase_pattern(end_role).finditer(text))
    if not start_matches or not end_matches:
        return False
    s, e = start_matches[0], end_matches[0]
    window = (s.end(), e.start()) if s.start() < e.start() else (e.end(), s.start())
    if window[0] >= window[1]:
        return False
    for m in find_matches(text, DIMLEX_LEXICON):
        if m["category"] in TEMPORAL_COMPARISON_CATS and m["start"] >= window[0] and m["end"] <= window[1]:
            return True
    return False


both_df = swap_target[swap_target["attribution"] == "both"].copy()
both_df["has_connective"] = both_df.apply(
    lambda r: connective_between_roles(r["justification"], r["voted_player_start_role"], r["voted_player_end_role"]),
    axis=1,
)
print(f"'both' instances: {len(both_df)}; with a linking Temporal/Comparison connective: {int(both_df['has_connective'].sum())}")

per_game_connective = both_df.groupby(["model", "decoding_group", "game_id"])["has_connective"].mean().reset_index()
connective_rows = []
for (mdl, dg), grp in per_game_connective.groupby(["model", "decoding_group"]):
    n_inst = int(both_df[(both_df["model"] == mdl) & (both_df["decoding_group"] == dg)].shape[0])
    mean, lo, hi = boot_mean(grp["has_connective"].values)
    connective_rows.append({
        "model": mdl, "decoding_group": dg, "n_games": len(grp), "n_instances": n_inst,
        "pct_with_connective": round(100 * mean, 1) if not np.isnan(mean) else np.nan,
        "ci_low": round(100 * lo, 1) if not np.isnan(lo) else np.nan,
        "ci_high": round(100 * hi, 1) if not np.isnan(hi) else np.nan,
    })
swap_narration_connective_check = pd.DataFrame(connective_rows)
print(swap_narration_connective_check)

'both' instances: 208; with a linking Temporal/Comparison connective: 120
  model decoding_group  n_games  n_instances  pct_with_connective  ci_low  ci_high
0    2B         Greedy        5            5                 60.0    20.0    100.0
1    2B     Stochastic       10           13                 50.0    20.0     80.0
2   31B         Greedy       32           32                 56.2    40.6     71.9
3   31B     Stochastic       48           95                 64.9    54.5     75.3
4    4B         Greedy       16           16                 56.2    31.2     81.2
5    4B     Stochastic       34           47                 63.2    47.5     78.4


### Chance-adjusted check -- is `both` just a stylistic trait, or targeted?

`both` (Section above) is a **binary presence** check: does the justification
mention the start role *anywhere* and the end role *anywhere*, regardless of
count. That leaves a real hole: 31B mentions role names at a much higher
density overall (~8.4 role mentions per 100 words vs. 2B's ~5.6 and 4B's
~4.4). A model that simply names more distinct roles per justification, for
stylistic reasons unrelated to this specific swap, would score higher on
`both` by sheer coverage -- not because it is tracking *this* player's
*actual* swap.

This computes an exact per-instance chance baseline: given how many of the
**11 other, irrelevant** roles a specific justification happens to mention
(`k'`), what is the probability that a *random* pair drawn from those same
11 roles would also land on "both mentioned," purely by the breadth of role
vocabulary that specific text uses? (Hypergeometric: population = 11 other
roles, `k'` of them are "hits" in this text, draw a random pair of size 2.)
`excess_both = is_both - expected_random_both` per instance is then the
co-mention rate *beyond* what this model's own general role-mention breadth
in that instance would already predict. A model whose `mean_excess_both` CI
clears 0 is doing more than sounding role-heavy -- it is landing on the
*correct* pair more than chance, given its own verbosity, would predict.

In [19]:
def expected_random_both(row):
    other_pool = ALL_ROLES - {row["voted_player_start_role"], row["voted_player_end_role"]}
    mentions = get_role_mentions(row["justification"])
    k_prime = len(mentions & other_pool)
    return chance_multi_match_prob(len(other_pool), k_prime, 2)


swap_target["expected_random_both"] = swap_target.apply(expected_random_both, axis=1)
swap_target["excess_both"] = swap_target["is_both"] - swap_target["expected_random_both"]

per_game_excess = (
    swap_target.groupby(["model", "decoding_group", "game_id"])
    .agg(observed=("is_both", "mean"), expected_random=("expected_random_both", "mean"), excess=("excess_both", "mean"))
    .reset_index()
)
chance_adjusted_rows = []
for (mdl, dg), grp in per_game_excess.groupby(["model", "decoding_group"]):
    n_inst = int(swap_target[(swap_target["model"] == mdl) & (swap_target["decoding_group"] == dg)].shape[0])
    mean_excess, lo, hi = boot_mean(grp["excess"].values)
    chance_adjusted_rows.append({
        "model": mdl, "decoding_group": dg, "n_games": len(grp), "n_instances": n_inst,
        "mean_observed_both": round(float(grp["observed"].mean()), 3),
        "mean_expected_random_both": round(float(grp["expected_random"].mean()), 3),
        "mean_excess_both": round(mean_excess, 3) if not np.isnan(mean_excess) else np.nan,
        "ci_low": round(lo, 3) if not np.isnan(lo) else np.nan,
        "ci_high": round(hi, 3) if not np.isnan(hi) else np.nan,
        "excludes_zero": bool(not np.isnan(lo) and (lo > 0 or hi < 0)),
        "inconclusive_at_this_n": bool(np.isnan(lo) or not (lo > 0 or hi < 0)),
    })
swap_chance_adjusted_check = pd.DataFrame(chance_adjusted_rows)
print(swap_chance_adjusted_check)

  model decoding_group  n_games  n_instances  mean_observed_both  mean_expected_random_both  mean_excess_both  ci_low  ci_high  \
0    2B         Greedy       67           67               0.075                      0.003             0.072   0.013    0.146   
1    2B     Stochastic       90          196               0.061                      0.002             0.059   0.023    0.102   
2   31B         Greedy       70           70               0.457                      0.030             0.427   0.311    0.550   
3   31B     Stochastic       87          223               0.420                      0.031             0.388   0.300    0.479   
4    4B         Greedy       81           81               0.198                      0.009             0.188   0.102    0.277   
5    4B     Stochastic      103          240               0.194                      0.010             0.184   0.123    0.249   

   excludes_zero  inconclusive_at_this_n  
0           True                   False  
1  

### Stratifying by whether the pair involves "Werewolf" -- and why this is a *different population*, not a stricter measure

The task *is* to identify the Werewolf, so the word appears in **94-99.5%** of all
justifications regardless of model or reasoning. Whenever "Werewolf" is one of the two
roles in the swap pair (~55% of cases, near-identically across models), that half of the
match is nearly free.

**Critical structural fact:** `voted_player_end_role == "Werewolf"` is **exactly
equivalent to `is_correct`** -- correctness is *defined* as voting for a player whose end
role is Werewolf (verified: 267/267 and 610/610, no exceptions). The swap-target set
therefore decomposes into three strata that differ in difficulty *and* in outcome:

| stratum | n | what it contains |
|---|---|---|
| `end == Werewolf` | 267 | **all the correct votes**; the end role is given away by the task framing |
| `start == Werewolf` only | 224 | "defensible misses" -- target began as Werewolf but swapped away |
| **neither role is Werewolf** | **386** | **the hardest case: both roles must be earned -- and contains _zero_ correct votes by construction** |

So the Werewolf-free subset is **not** a conservative recomputation of the same quantity.
It is the hardest stratum, and it consists entirely of votes the model got **wrong**.
That makes it a genuinely informative measure -- *does the model track the night mechanics
even when it reaches the wrong conclusion?* -- but it must be reported as such, not as a
stricter version of the headline number. Both strata are reported below.

In [20]:
swap_target["ww_end"] = swap_target["voted_player_end_role"].eq("Werewolf")
swap_target["ww_start"] = swap_target["voted_player_start_role"].eq("Werewolf")
swap_target["ww_in_pair"] = swap_target["ww_end"] | swap_target["ww_start"]

# Verify the structural equivalence that makes this a stratification, not a filter.
_x = pd.crosstab(swap_target["ww_end"], swap_target["is_correct"])
print("end_role == 'Werewolf'  vs  is_correct  (off-diagonals must be zero):")
print(_x)
assert _x.values[0, 1] == 0 and _x.values[1, 0] == 0, "end==WW should be identical to is_correct"

print("\nshare of swap pairs involving 'Werewolf' (similar across models -> no differential selection):")
print(swap_target.groupby("model")["ww_in_pair"].mean().round(3))
print("\naccuracy within each stratum (the Werewolf-free stratum is 0% correct BY CONSTRUCTION):")
print(swap_target.groupby(["model", "ww_in_pair"])["is_correct"].agg(["mean", "size"]).round(3))

clean = swap_target[~swap_target["ww_in_pair"]]
# Recompute the chance baseline ON THIS SUBSET -- the baseline in the previous section was
# computed over all swap targets, so mixing the two would compare different populations.
# Note the null correctly accounts for "Werewolf" still being in the 11-role other-pool
# here: nearly every justification mentions it, which raises k' and so raises the baseline.
per_game_clean = (
    clean.groupby(["model", "decoding_group", "game_id"])
    .agg(share_both=("is_both", "mean"),
         expected_random=("expected_random_both", "mean"),
         excess=("excess_both", "mean"),
         n_instances=("is_both", "size"))
    .reset_index()
)
rows = []
for (mdl, dg), grp in per_game_clean.groupby(["model", "decoding_group"]):
    mean, lo, hi = boot_mean(grp["share_both"].values)
    ex_mean, ex_lo, ex_hi = boot_mean(grp["excess"].values)
    rows.append({
        "model": mdl, "decoding_group": dg,
        "n_games": len(grp), "n_instances": int(grp["n_instances"].sum()),
        "share_both_excl_werewolf": round(mean, 3) if not np.isnan(mean) else np.nan,
        "ci_low": round(lo, 3) if not np.isnan(lo) else np.nan,
        "ci_high": round(hi, 3) if not np.isnan(hi) else np.nan,
        "chance_baseline_this_subset": round(float(grp["expected_random"].mean()), 3),
        "excess_over_chance": round(ex_mean, 3) if not np.isnan(ex_mean) else np.nan,
        "excess_ci_low": round(ex_lo, 3) if not np.isnan(ex_lo) else np.nan,
        "excess_ci_high": round(ex_hi, 3) if not np.isnan(ex_hi) else np.nan,
        "excess_excludes_zero": bool(not np.isnan(ex_lo) and (ex_lo > 0 or ex_hi < 0)),
    })
swap_no_werewolf_check = pd.DataFrame(rows)
print("\nshare_both on pairs NOT involving 'Werewolf' (the conservative figure to quote):")
print(swap_no_werewolf_check.to_string(index=False))

cross = pd.concat(
    [unpaired_cross_model_ci(per_game_clean, "share_both", dg) for dg in DECODING_ORDER], ignore_index=True
)
print("\nCross-model contrasts on the Werewolf-free subset:")
print(cross.to_string(index=False))
swap_no_werewolf_bootstrap_ci = cross

end_role == 'Werewolf'  vs  is_correct  (off-diagonals must be zero):
is_correct  False  True 
ww_end                  
False         610      0
True            0    267

share of swap pairs involving 'Werewolf' (similar across models -> no differential selection):
model
2B     0.551
31B    0.584
4B     0.545
Name: ww_in_pair, dtype: float64

accuracy within each stratum (the Werewolf-free stratum is 0% correct BY CONSTRUCTION):
                      mean  size
model ww_in_pair                
2B    False            0.0   118
      True        0.544828   145
31B   False            0.0   122
      True        0.555556   171
4B    False            0.0   146
      True        0.531429   175

share_both on pairs NOT involving 'Werewolf' (the conservative figure to quote):
model decoding_group  n_games  n_instances  share_both_excl_werewolf  ci_low  ci_high  chance_baseline_this_subset  excess_over_chance  excess_ci_low  excess_ci_high  excess_excludes_zero
   2B         Greedy       31    


Cross-model contrasts on the Werewolf-free subset:
decoding_group model_a model_b  n_games_a  n_games_b  mean_a  mean_b   diff  ci_low  ci_high  excludes_zero  inconclusive_at_this_n
    Stochastic     31B      2B         42         46   0.290   0.036  0.253   0.129    0.390           True                   False
    Stochastic     31B      4B         42         51   0.290   0.118  0.172   0.031    0.316           True                   False
    Stochastic      4B      2B         51         46   0.118   0.036  0.081   0.003    0.163           True                   False
        Greedy     31B      2B         29         31   0.276   0.065  0.211   0.037    0.414           True                   False
        Greedy     31B      4B         29         38   0.276   0.053  0.223   0.051    0.404           True                   False
        Greedy      4B      2B         38         31   0.053   0.065 -0.012  -0.129    0.099          False                    True


# Analysis 6 -- Ground-truth claim-contradiction flagging

> ## ★ Corroborating evidence for RQ1 (independent of Analysis 5)
>
> Analysis 5 tests tracking of *card movement*. This tests tracking of *what players
> said* -- the other half of ONUW deduction. Using the DeepSeek identity-claim
> annotations, we know which players genuinely claimed more than one role during a game.
>
> **The test:** when the model votes for such a player, does its justification name **two
> or more** of that player's actual claims -- i.e. did it notice the contradiction?
>
> ### ⚠ Construct-validity correction -- read before interpreting
>
> `is_self_contradiction` is computed as a **pure set intersection** over all of a
> player's role claims across the game, with **no notion of time or card swaps**. But in
> ONUW roles legitimately *change*: a Robber swaps and then looks at its new card, so
> "I was the Robber, now I'm the Seer" is **correct play**, not a lie -- yet
> `{Robber} & {Seer} = {}` flags it as a contradiction.
>
> **60% of flagged players had a role that genuinely changed during the night**, and the
> most common flagged patterns are exactly `Robber -> Werewolf`, `Robber -> Seer`,
> `Robber -> Tanner`. The unrestricted metric therefore conflates *detecting a lie* with
> *narrating a swap* -- and consequently **56% of its instances are also Analysis 5
> instances**, so it does **not** provide independent corroboration of Analysis 5.
>
> **Fix applied here:** the primary analysis is restricted to players whose role did
> **not** change, where a contradictory claim has no swap explanation and is a genuine
> inconsistency. This is construct-valid and disjoint from Analysis 5 by construction.
> The unrestricted version is retained below, explicitly labelled as confounded.
>
> **Cost:** n drops from 411 to 181 instances (13-22 games per model per decoding cell).
> This is thin for a game-level bootstrap; expect wide CIs and treat any cell with few
> games as underpowered rather than null.

The claim-consistency analog to Analysis 5. `player_conflict_features.csv`
(DeepSeek-annotated from the identity-claim transcripts) flags players who
claimed **more than one distinct role** over the course of a game
(`is_self_contradiction`), with the actual set of claimed roles in
`roles_claimed`. This checks whether the justification, when it votes for
one of these self-contradicting players, explicitly names **two or more**
of that specific player's own claimed roles -- evidence the model noticed
the contradiction, not just that it mentions roles in general. Uses the same
`ROLE_ATTRIBUTION_PATTERN` (13-role vocabulary) already built for Analysis 5.

**Caveat specific to this analysis, on top of the usual co-occurrence
caveat:** the ground truth here (`is_self_contradiction`, `roles_claimed`)
is DeepSeek-annotated, not human-verified like the `start_roles`/`end_roles`
ground truth Analysis 5 uses. Errors in DeepSeek's claim extraction would
propagate directly into this diagnostic subset and its classification.

In [21]:
contradiction_target_all = VOTES_TARGETED[
    VOTES_TARGETED.apply(
        lambda r: r["chosen_player_name"] in self_contradiction_by_game.get(r["game_id"], set()), axis=1
    )
].copy()
# A flagged "contradiction" is only genuine if the player's role did NOT change --
# otherwise the two claims can both be true at different times (legitimate swap narration).
contradiction_target_all["role_changed"] = (
    contradiction_target_all["voted_player_start_role"] != contradiction_target_all["voted_player_end_role"]
)
contradiction_target = contradiction_target_all[~contradiction_target_all["role_changed"]].copy()

n_all, n_gen = len(contradiction_target_all), len(contradiction_target)
print(f"Votes targeting a flagged self-contradicting claimant: {n_all} / {len(VOTES_TARGETED)}")
print(f"  ...of which the player's role ACTUALLY CHANGED (swap, not a lie): {n_all - n_gen} ({100*(n_all-n_gen)/n_all:.0f}%)")
print(f"  ...GENUINE contradictions used as the primary subset:            {n_gen} ({100*n_gen/n_all:.0f}%)")
print("\nPrimary (genuine-contradiction) subset, instances per cell:")
print(contradiction_target.groupby(["model", "decoding_group"]).size().unstack(fill_value=0))
print("\nGames per cell (the bootstrap unit -- small; see the underpowered caveat above):")
print(contradiction_target.groupby(["model", "decoding_group"])["game_id"].nunique().unstack(fill_value=0))


def classify_contradiction_flagging(row):
    claimed = claimed_roles_by_game_player.get((row["game_id"], row["chosen_player_name"]), frozenset()) & ALL_ROLES
    mentions = get_role_mentions(row["justification"])
    matched = claimed & mentions
    if len(matched) >= 2:
        return "flags_multiple_claims"
    if len(matched) == 1:
        return "mentions_one_claim"
    if mentions:
        return "mentions_other_role"
    return "no_mention"


contradiction_target["attribution"] = contradiction_target.apply(classify_contradiction_flagging, axis=1)
contradiction_attribution_counts = (
    contradiction_target.groupby(["model", "decoding_group", "attribution"]).size().unstack(fill_value=0)
)
print("\nContradiction-flagging classification:")
print(contradiction_attribution_counts)

print("\nAvg n_words on this diagnostic subset (verbosity check):")
print(contradiction_target.groupby("model")["n_words"].mean().round(1))

# Confounded comparison: the unrestricted subset, including swap-explained "contradictions".
contradiction_target_all["attribution"] = contradiction_target_all.apply(classify_contradiction_flagging, axis=1)
contradiction_confounded_comparison = (
    pd.concat([
        (contradiction_target.assign(subset="genuine_only")
         .groupby(["subset", "model", "decoding_group"])
         .apply(lambda d: (d["attribution"] == "flags_multiple_claims").mean(), include_groups=False)),
        (contradiction_target_all.assign(subset="confounded_all")
         .groupby(["subset", "model", "decoding_group"])
         .apply(lambda d: (d["attribution"] == "flags_multiple_claims").mean(), include_groups=False)),
    ]).rename("share_flagged").reset_index()
)
print("\nGenuine-only vs confounded-all (the latter mixes in swap narration -- do NOT report it):")
print(contradiction_confounded_comparison.pivot(index=["model", "decoding_group"],
                                                columns="subset", values="share_flagged").round(3))

Votes targeting a flagged self-contradicting claimant: 411 / 1963
  ...of which the player's role ACTUALLY CHANGED (swap, not a lie): 230 (56%)
  ...GENUINE contradictions used as the primary subset:            181 (44%)

Primary (genuine-contradiction) subset, instances per cell:
decoding_group  Greedy  Stochastic
model                             
2B                  13          36
31B                 16          44
4B                  21          51

Games per cell (the bootstrap unit -- small; see the underpowered caveat above):
decoding_group  Greedy  Stochastic
model                             
2B                  13          20
31B                 16          19
4B                  21          22

Contradiction-flagging classification:
attribution           flags_multiple_claims  mentions_one_claim  mentions_other_role
model decoding_group                                                                
2B    Greedy                              3                   7             


Genuine-only vs confounded-all (the latter mixes in swap narration -- do NOT report it):
subset                confounded_all  genuine_only
model decoding_group                              
2B    Greedy                   0.172         0.231
      Stochastic               0.151         0.139
31B   Greedy                   0.758         0.812
      Stochastic               0.680         0.750
4B    Greedy                   0.500         0.429
      Stochastic               0.504         0.451


In [22]:
contradiction_target["is_flagged"] = (contradiction_target["attribution"] == "flags_multiple_claims").astype(float)
per_game_flagged = (
    contradiction_target.groupby(["model", "decoding_group", "game_id"])
    .agg(share_flagged=("is_flagged", "mean"), n_instances=("is_flagged", "size"))
    .reset_index()
)
share_flagged_by_model = (
    per_game_flagged.groupby(["model", "decoding_group"])
    .agg(n_games=("game_id", "nunique"), n_instances=("n_instances", "sum"), mean_share_flagged=("share_flagged", "mean"))
    .reset_index()
)
print(share_flagged_by_model.round(3))

contradiction_bootstrap_ci = pd.concat(
    [unpaired_cross_model_ci(per_game_flagged, "share_flagged", dg) for dg in DECODING_ORDER], ignore_index=True
)
print("\nCross-model contrast on share_flagged (unpaired):")
print(contradiction_bootstrap_ci)

  model decoding_group  n_games  n_instances  mean_share_flagged
0    2B         Greedy       13           13               0.231
1    2B     Stochastic       20           36               0.167
2   31B         Greedy       16           16               0.812
3   31B     Stochastic       19           44               0.728
4    4B         Greedy       21           21               0.429
5    4B     Stochastic       22           51               0.424

Cross-model contrast on share_flagged (unpaired):
  decoding_group model_a model_b  n_games_a  n_games_b  mean_a  mean_b   diff  ci_low  ci_high  excludes_zero  inconclusive_at_this_n
0     Stochastic     31B      2B         19         20   0.728   0.167  0.561   0.318    0.777           True                   False
1     Stochastic     31B      4B         19         22   0.728   0.424  0.304   0.041    0.551           True                   False
2     Stochastic      4B      2B         22         20   0.424   0.167  0.258   0.028    0.4

### Precision layer -- DiMLex-connective check (reuses Analysis 5's function)

Among `flags_multiple_claims` instances, does a DiMLex Temporal/Comparison
marker sit between the two claimed-role mentions (e.g. "first said Seer but
later claimed Robber")? Same `connective_between_roles` function as
Analysis 5, applied to the first two matched claimed roles in alphabetical
order (deterministic when more than two are claimed).

In [23]:
flagged_df = contradiction_target[contradiction_target["attribution"] == "flags_multiple_claims"].copy()


def contradiction_connective_check(row):
    claimed = claimed_roles_by_game_player.get((row["game_id"], row["chosen_player_name"]), frozenset()) & ALL_ROLES
    mentions = get_role_mentions(row["justification"])
    matched = sorted(claimed & mentions)
    if len(matched) < 2:
        return False
    return connective_between_roles(row["justification"], matched[0], matched[1])


flagged_df["has_connective"] = flagged_df.apply(contradiction_connective_check, axis=1)
print(f"'flags_multiple_claims' instances: {len(flagged_df)}; with a linking connective: {int(flagged_df['has_connective'].sum())}")

per_game_contradiction_connective = flagged_df.groupby(["model", "decoding_group", "game_id"])["has_connective"].mean().reset_index()
contradiction_connective_rows = []
for (mdl, dg), grp in per_game_contradiction_connective.groupby(["model", "decoding_group"]):
    n_inst = int(flagged_df[(flagged_df["model"] == mdl) & (flagged_df["decoding_group"] == dg)].shape[0])
    mean, lo, hi = boot_mean(grp["has_connective"].values)
    contradiction_connective_rows.append({
        "model": mdl, "decoding_group": dg, "n_games": len(grp), "n_instances": n_inst,
        "pct_with_connective": round(100 * mean, 1) if not np.isnan(mean) else np.nan,
        "ci_low": round(100 * lo, 1) if not np.isnan(lo) else np.nan,
        "ci_high": round(100 * hi, 1) if not np.isnan(hi) else np.nan,
    })
contradiction_connective_check_table = pd.DataFrame(contradiction_connective_rows)
print(contradiction_connective_check_table)

'flags_multiple_claims' instances: 86; with a linking connective: 46
  model decoding_group  n_games  n_instances  pct_with_connective  ci_low  ci_high
0    2B         Greedy        3            3                 66.7     0.0    100.0
1    2B     Stochastic        5            5                 60.0    20.0    100.0
2   31B         Greedy       13           13                 61.5    30.8     84.6
3   31B     Stochastic       15           33                 57.8    35.6     80.0
4    4B         Greedy        9            9                 55.6    22.2     88.9
5    4B     Stochastic       12           23                 54.2    33.3     76.4


### Chance-adjusted check -- same control as Analysis 5

Same logic as Analysis 5's chance-adjusted check, generalized to however many
distinct roles (`m`) this specific player actually claimed (usually 2, the
hypergeometric formula handles more). Controls for the same concern: a model
that names roles more often in general would flag more contradictions by
coincidence, not because it specifically noticed this player's claims
conflicted.

In [24]:
def expected_random_flag(row):
    claimed = claimed_roles_by_game_player.get((row["game_id"], row["chosen_player_name"]), frozenset()) & ALL_ROLES
    m = len(claimed)
    other_pool = ALL_ROLES - claimed
    mentions = get_role_mentions(row["justification"])
    k_prime = len(mentions & other_pool)
    return chance_multi_match_prob(len(other_pool), k_prime, m)


contradiction_target["expected_random_flag"] = contradiction_target.apply(expected_random_flag, axis=1)
contradiction_target["excess_flag"] = contradiction_target["is_flagged"] - contradiction_target["expected_random_flag"]

per_game_excess_flag = (
    contradiction_target.groupby(["model", "decoding_group", "game_id"])
    .agg(observed=("is_flagged", "mean"), expected_random=("expected_random_flag", "mean"), excess=("excess_flag", "mean"))
    .reset_index()
)
contradiction_chance_adjusted_rows = []
for (mdl, dg), grp in per_game_excess_flag.groupby(["model", "decoding_group"]):
    n_inst = int(contradiction_target[(contradiction_target["model"] == mdl) & (contradiction_target["decoding_group"] == dg)].shape[0])
    mean_excess, lo, hi = boot_mean(grp["excess"].values)
    contradiction_chance_adjusted_rows.append({
        "model": mdl, "decoding_group": dg, "n_games": len(grp), "n_instances": n_inst,
        "mean_observed_flagged": round(float(grp["observed"].mean()), 3),
        "mean_expected_random_flagged": round(float(grp["expected_random"].mean()), 3),
        "mean_excess_flagged": round(mean_excess, 3) if not np.isnan(mean_excess) else np.nan,
        "ci_low": round(lo, 3) if not np.isnan(lo) else np.nan,
        "ci_high": round(hi, 3) if not np.isnan(hi) else np.nan,
        "excludes_zero": bool(not np.isnan(lo) and (lo > 0 or hi < 0)),
        "inconclusive_at_this_n": bool(np.isnan(lo) or not (lo > 0 or hi < 0)),
    })
contradiction_chance_adjusted_check = pd.DataFrame(contradiction_chance_adjusted_rows)
print(contradiction_chance_adjusted_check)

  model decoding_group  n_games  n_instances  mean_observed_flagged  mean_expected_random_flagged  mean_excess_flagged  ci_low  ci_high  \
0    2B         Greedy       13           13                  0.231                         0.003                0.228  -0.003    0.462   
1    2B     Stochastic       20           36                  0.167                         0.009                0.158   0.034    0.312   
2   31B         Greedy       16           16                  0.812                         0.094                0.719   0.518    0.883   
3   31B     Stochastic       19           44                  0.728                         0.040                0.688   0.499    0.859   
4    4B         Greedy       21           21                  0.429                         0.010                0.418   0.227    0.616   
5    4B     Stochastic       22           51                  0.424                         0.006                0.418   0.238    0.597   

   excludes_zero  inconclu

# Analysis 7 -- Does the crowd-conformity effect reduce to shared social-salience susceptibility?

> ## ★ Where RQ1 and RQ2 meet
>
> The models **never see the human votes** -- so "persuaded by the crowd" cannot be
> literal; there is no channel. The live hypothesis is that models and humans read the
> *same transcript* and are pulled toward the *same* misleading signal.
>
> **The test:** on dissociated games, is the social-salience proxy better at explaining
> **herded** votes (the crowd's wrong pick) than **independent** ones (the true
> Werewolf)? A model with only one mechanism should show no gap -- salience explains its
> hits and misses equally. A model with a second, evidence-based route should show a
> large gap.
>
> ## ⚠ NEGATIVE RESULT -- this analysis does not support a behavioural conclusion
>
> The raw gaps look striking (31B: 0.478 greedy, 0.260 stochastic; 2B: ~0.07-0.10) and
> initially appeared to show that 31B has a second, evidence-based route to the answer.
> **A proper null baseline dissolves the entire effect.** After subtracting the structural
> overlap, no model's excess gap clears zero -- 31B's drops from 0.478 to **0.061**.
>
> **Why.** Humans vote on the basis of accusations too, so the crowd's modal target is
> frequently the *same player* as the most-accused player (44% of dissociated games),
> whereas the true Werewolf rarely is (33%). Worse, **72% of dissociated games have
> exactly one crowd-modal target and exactly one Werewolf** -- so a model that votes
> "herded" had no choice about *which* player that was. Observed and baseline become
> identical by construction, and the models cancel out.
>
> The contrast therefore measures **a property of the dataset** -- how often the crowd's
> pick coincides with the most-accused player, versus how often the Werewolf does -- and
> not model behaviour. It is retained here as a documented negative result and a caution
> about confounded contrasts, not as evidence for either RQ.
>
> **Read `excess_gap`, never `raw_gap`.** RQ1's positive evidence is in Analyses 5-6,
> which compare a model against ground truth rather than against a class defined by the
> very signal being tested.

RQ2 was originally framed as "persuaded by the crowd," but the models never
see the human crowd's votes -- there is no channel for literal persuasion.
The live hypothesis is narrower and more mechanistic: models and the human
crowd are both reading the *same* transcript, and both may be drawn to the
same misleading surface signal (heavy accusation, a confident self-claim).
If so, a "herded" vote (matches the crowd's wrong pick, Analysis 2) should
line up with the surrogate's social-salience proxy (`is_predictable`,
Analysis 1 -- most-accused player or self-claimed Werewolf) *more* than an
"independent" (correct) vote does. This reuses `V1`'s `is_predictable` and
`V2`'s `vote_class`, joined by `justification_id` -- no new ground truth.

In [25]:
V7_full = V2.merge(V1[["justification_id", "is_predictable"]], on="justification_id", how="left")
V7 = V7_full[V7_full["vote_class"] != "other"].copy()
V7["is_herded"] = V7["vote_class"].eq("herded")

print("Overall (pooled across models): share of votes matching the social-salience proxy, by vote class")
print(V7.groupby("vote_class")["is_predictable"].agg(["mean", "sum", "size"]).round(3))

salience_reliance_counts = (
    V7.groupby(["model", "decoding_group", "vote_class", "is_predictable"]).size().unstack(fill_value=0)
)
print("\nBy model:")
print(salience_reliance_counts)

salience_reliance_gap_ci = contrast_table(V7, "is_predictable", "is_herded", "herded", "independent")
print("\nSalience-proxy match rate: herded vs. independent votes (per model, per-game weighted, bootstrap CI):")
print(salience_reliance_gap_ci)

Overall (pooled across models): share of votes matching the social-salience proxy, by vote class
              mean  sum  size
vote_class                   
herded       0.835  198   237
independent  0.652   86   132

By model:
is_predictable                    False  True 
model decoding_group vote_class               
2B    Greedy         herded           3     14
                     independent      3      9
      Stochastic     herded          11     46
                     independent      8     27
31B   Greedy         herded           2     17
                     independent      7      5
      Stochastic     herded           9     58
                     independent     11     15
4B    Greedy         herded           2     17
                     independent      5      6
      Stochastic     herded          12     46
                     independent     12     24

Salience-proxy match rate: herded vs. independent votes (per model, per-game weighted, bootstrap CI):
  model dec

### Probability-weighted companion

Same reasoning as Analysis 2's weighted companion: a game where the model's
stochastic runs split 2-1 between herded and independent should count 0.67
toward one arm and 0.33 toward the other, not full weight in both. Computed
on `V7_full` (all three vote classes retained, so weights are genuine shares
of each game's total valid instances).

In [26]:
salience_reliance_gap_weighted_ci = weighted_contrast_table(V7_full, "is_predictable", "vote_class", "herded", "independent")
print(salience_reliance_gap_weighted_ci)

  model decoding_group  n_games  weighted_mean_herded  weighted_mean_independent   diff  ci_low  ci_high  excludes_zero  \
0    2B         Greedy       40                 0.824                      0.750  0.074  -0.233    0.395          False   
1    2B     Stochastic       46                 0.826                      0.699  0.127  -0.118    0.388          False   
2   31B         Greedy       42                 0.895                      0.417  0.478   0.150    0.795           True   
3   31B     Stochastic       45                 0.858                      0.556  0.303  -0.011    0.631          False   
4    4B         Greedy       45                 0.895                      0.545  0.349   0.014    0.692           True   
5    4B     Stochastic       47                 0.800                      0.667  0.133  -0.159    0.440          False   

   inconclusive_at_this_n  
0                    True  
1                    True  
2                   False  
3                    True 

### Baseline-corrected gap -- controlling for structural overlap

**The raw gap above is inflated by a definitional overlap, and must not be read against
zero.** Humans also vote on the basis of accusations, so **the crowd's modal target is
frequently the same player as the most-accused player**. A "herded" vote therefore counts
as *predictable* partly by construction, independently of anything the model did.

The correct null is not zero but: **if the model picked an arbitrary member of the
relevant class, how predictable would it look anyway?** For each game:

- `baseline_herded` = fraction of that game's crowd-modal targets that satisfy the proxy
- `baseline_independent` = fraction of that game's true Werewolves that satisfy the proxy

`excess = observed - baseline` per arm, then the two arms are contrasted exactly as before.
A model whose **excess gap** clears zero is relying on social salience for its herded votes
*more than the structure of the data already forces*. This is the same logic as the
chance-adjusted checks in Analyses 5-6, applied to the RQ1/RQ2 junction.

In [27]:
def proxy_hits(players, game_id):
    ma = most_accused_by_game.get(game_id)
    claimers = claims_werewolf_by_game.get(game_id, set())
    players = [p for p in players if p is not None]
    if not players:
        return np.nan
    return float(np.mean([1.0 if (p == ma or p in claimers) else 0.0 for p in players]))


baseline_rows = []
for game_id in sorted({g for g in V7_full["game_id"].unique()}):
    crowd_targets = village_targets_by_game.get(game_id, set())
    baseline_rows.append({
        "game_id": game_id,
        "baseline_herded": proxy_hits(sorted(crowd_targets), game_id),
    })
baseline_h = pd.DataFrame(baseline_rows).set_index("game_id")["baseline_herded"].to_dict()

# wolves are per (model, game) in the source table, though identical across models
baseline_i = {}
for (mdl, gid), wolves in wolves_by_model_game.items():
    baseline_i[(mdl, gid)] = proxy_hits(sorted(wolves), gid)

per_game_obs = (
    V7_full[V7_full["vote_class"].isin(["herded", "independent"])]
    .groupby(["model", "decoding_group", "vote_class", "game_id"])["is_predictable"]
    .mean().reset_index()
)
per_game_obs["baseline"] = [
    baseline_h.get(r.game_id, np.nan) if r.vote_class == "herded"
    else baseline_i.get((r.model, r.game_id), np.nan)
    for r in per_game_obs.itertuples(index=False)
]
per_game_obs["excess"] = per_game_obs["is_predictable"] - per_game_obs["baseline"]
per_game_obs = per_game_obs.dropna(subset=["excess"])

rows = []
for (mdl, dg), grp in per_game_obs.groupby(["model", "decoding_group"]):
    h = grp[grp["vote_class"] == "herded"]
    i = grp[grp["vote_class"] == "independent"]
    raw_gap = h["is_predictable"].mean() - i["is_predictable"].mean()
    structural = h["baseline"].mean() - i["baseline"].mean()
    diff, lo, hi = boot_diff(h["excess"].values, i["excess"].values)
    rows.append({
        "model": mdl, "decoding_group": dg,
        "n_games_herded": len(h), "n_games_independent": len(i),
        "raw_gap": round(raw_gap, 3),
        "structural_baseline_gap": round(structural, 3),
        "excess_gap": round(diff, 3) if not np.isnan(diff) else np.nan,
        "ci_low": round(lo, 3) if not np.isnan(lo) else np.nan,
        "ci_high": round(hi, 3) if not np.isnan(hi) else np.nan,
        "excludes_zero": bool(not np.isnan(lo) and (lo > 0 or hi < 0)),
        "inconclusive_at_this_n": bool(np.isnan(lo) or not (lo > 0 or hi < 0)),
    })
salience_baseline_corrected_ci = pd.DataFrame(rows)
print(salience_baseline_corrected_ci.to_string(index=False))
print("\nread: raw_gap includes structural overlap; excess_gap is the part attributable")
print("to model behaviour. Compare excess_gap (not raw_gap) against zero.")

model decoding_group  n_games_herded  n_games_independent  raw_gap  structural_baseline_gap  excess_gap  ci_low  ci_high  excludes_zero  inconclusive_at_this_n
   2B         Greedy              17                   12    0.074                    0.047       0.027  -0.086    0.147          False                    True
   2B     Stochastic              28                   19    0.095                    0.112      -0.017  -0.111    0.071          False                    True
  31B         Greedy              19                   12    0.478                    0.417       0.061   0.000    0.158          False                    True
  31B     Stochastic              29                   15    0.260                    0.253       0.007  -0.077    0.086          False                    True
   4B         Greedy              19                   11    0.349                    0.360      -0.010  -0.144    0.114          False                    True
   4B     Stochastic              25    

# Save outputs and final sanity check

In [28]:
preexisting_files = {f.name for f in OUTPUT_DIR.glob("*.csv")}

outputs = {
    "04a_contingency_by_surrogate_predictability.csv": surrogate_means,
    "04a_contingency_by_surrogate_predictability_bootstrap_ci.csv": surrogate_contingency_ci,
    "04b_markers_by_herded_independent.csv": vote_class_counts.reset_index(),
    "04b_markers_by_herded_independent_bootstrap_ci.csv": markers_by_herded_independent_ci,
    "04b_markers_by_herded_independent_weighted_bootstrap_ci.csv": markers_by_herded_independent_weighted_ci,
    "04b_excluded_other_votes.csv": excluded_other,
    "04d_mention_check_faithfulness.csv": mention_check,
    "05a_swap_target_role_attribution.csv": attribution_counts.reset_index(),
    "05a_swap_target_role_attribution_bootstrap_ci.csv": swap_target_bootstrap_ci,
    "05b_swap_narration_connective_check.csv": swap_narration_connective_check,
    "06a_contradiction_flagging.csv": contradiction_attribution_counts.reset_index(),
    "06a_contradiction_flagging_bootstrap_ci.csv": contradiction_bootstrap_ci,
    "06b_contradiction_connective_check.csv": contradiction_connective_check_table,
    "05c_swap_chance_adjusted_check.csv": swap_chance_adjusted_check,
    "06c_contradiction_chance_adjusted_check.csv": contradiction_chance_adjusted_check,
    "07a_salience_reliance_gap.csv": salience_reliance_counts.reset_index(),
    "07a_salience_reliance_gap_bootstrap_ci.csv": salience_reliance_gap_ci,
    "07a_salience_reliance_gap_weighted_bootstrap_ci.csv": salience_reliance_gap_weighted_ci,
    "07b_salience_baseline_corrected_ci.csv": salience_baseline_corrected_ci,
    "00_feature_availability.csv": feature_availability_summary,
    "00_circle_vote_selection.csv": circle_vote_selection,
    "05d_swap_excl_werewolf_check.csv": swap_no_werewolf_check,
    "05d_swap_excl_werewolf_bootstrap_ci.csv": swap_no_werewolf_bootstrap_ci,
    "06d_contradiction_genuine_vs_confounded.csv": contradiction_confounded_comparison,
}

foreign_files = {
    f for f in preexisting_files
    if not f.startswith(("00_", "04a", "04b", "04c", "04d", "05a", "05b", "05c", "05d",
                         "06a", "06b", "06c", "06d", "07a", "07b"))
}
new_collisions = set(outputs) & foreign_files
assert not new_collisions, f"Filename collision with pre-existing DiMLex-extension outputs: {new_collisions}"

for name, df in outputs.items():
    df.to_csv(OUTPUT_DIR / name, index=False, encoding="utf-8-sig")
    assert len(df) > 0, f"{name} is empty"

print("Saved (new):")
for name in outputs:
    print(" -", name)
print("\nPre-existing DiMLex-extension files in the same directory (untouched):")
for name in sorted(preexisting_files):
    print(" -", name)

Saved (new):
 - 04a_contingency_by_surrogate_predictability.csv
 - 04a_contingency_by_surrogate_predictability_bootstrap_ci.csv
 - 04b_markers_by_herded_independent.csv
 - 04b_markers_by_herded_independent_bootstrap_ci.csv
 - 04b_markers_by_herded_independent_weighted_bootstrap_ci.csv
 - 04b_excluded_other_votes.csv
 - 04d_mention_check_faithfulness.csv
 - 05a_swap_target_role_attribution.csv
 - 05a_swap_target_role_attribution_bootstrap_ci.csv
 - 05b_swap_narration_connective_check.csv
 - 06a_contradiction_flagging.csv
 - 06a_contradiction_flagging_bootstrap_ci.csv
 - 06b_contradiction_connective_check.csv
 - 05c_swap_chance_adjusted_check.csv
 - 06c_contradiction_chance_adjusted_check.csv
 - 07a_salience_reliance_gap.csv
 - 07a_salience_reliance_gap_bootstrap_ci.csv
 - 07a_salience_reliance_gap_weighted_bootstrap_ci.csv
 - 07b_salience_baseline_corrected_ci.csv
 - 00_feature_availability.csv
 - 00_circle_vote_selection.csv
 - 05d_swap_excl_werewolf_check.csv
 - 05d_swap_excl_werewolf

# Findings: what this notebook establishes about the research questions

## RQ1 -- Do the models reason deductively about game rules?

**Yes for 31B, largely no for 2B, intermediate for 4B -- and this is invisible in
accuracy.** All three models score ~0.41 on the vote itself. The difference appears only
when you check whether their *stated reasoning is factually correct about the game state*:

| ground-truth test | 2B | 4B | 31B |
|---|---|---|---|
| correctly narrates a real role swap (A5) | 6-8% | ~19% | **42-46%** |
| catches a real self-contradiction (A6) | 14-17% | ~50% | **67-76%** |

Two independent ground truths, two different mechanisms, same ordering. Both survive a
**verbosity control** (4B writes the *longest* justifications yet tracks less than 31B)
and an exact **chance-adjusted control** (the coincidence baseline is <=3.1% everywhere;
the excess clears zero for every model). A third, fully independent instrument -- the
`preliminary_eval` rules questionnaire -- produces the same ordering, with 31B perfect and
stable and the failures of the smaller models concentrated in exactly the temporal
state-tracking items this notebook measures behaviourally.

**Crucially, the models do not differ in how they *express* a deduction, only in whether
they reach one.** Among justifications that do narrate a swap or flag a contradiction, the
rate of linking the two mentions with an explicit connective is ~50-65% for *all three*
models, with overlapping CIs.

## RQ2 -- Are the judgements driven by social pressure?

**Yes, for all three models, and rule competence provides no protection.**

- The vote is largely predicted by two crude social-salience features -- who was accused,
  who self-claimed (surrogate model, 0.41-0.56 top-1 vs 0.23 random).
- On games where the human crowd was wrong, all models put substantially more vote mass on
  the crowd's wrong pick than on the true Werewolf -- and **31B does so most of all**
  (2.58x, vs 1.66x for 2B and 1.61x for 4B). Better rule knowledge did *not* confer
  resistance.
- This is **not conformity**: the models never see the human votes. It is independent
  convergence on the same misleading transcript signal.
- Models are **transparent** about it (Analysis 4, ~98-100% mention rate) -- they follow
  salience openly rather than concealing it.
- The effect leaves **no trace in the language** (Analyses 1-2 are essentially null). It
  lives in the behaviour, not the prose.

## How the two answers fit together -- and what remains open

The two findings sit side by side without contradiction: 31B is measurably better at
rule-based deduction (RQ1) *and* is at least as susceptible to social salience as the
smaller models (RQ2). Rule competence is an **additional capability, not a safeguard**.

**What could not be established.** Analysis 7 attempted to connect the two -- to show that
31B's correct votes come from a *different mechanism* than its crowd-following errors.
That attempt **failed on methodological grounds**: the herded-vs-independent contrast is
confounded, because the crowd's modal target and the most-accused player are largely the
same person (44% overlap), and in 72% of the relevant games each class contains exactly
one player, leaving the model no within-class choice. Once the structural baseline is
subtracted, no model shows a significant excess. **The question of whether 31B's successes
and failures arise from different mechanisms therefore remains open**, and would need a
design in which class membership is not defined by the same signal being tested.

## Methodological finding (worth reporting in its own right)

**Some lexical proxies carry real signal and others do not -- and which is which cannot be
known a priori.** This is a more careful claim than "counting words fails", and the data
support the narrower version:

- **Temporal markers work.** 31B uses ~3x 2B's density, tracking the capability ordering
  that the ground-truth analyses independently confirm.
- **Contingency markers ("because", "therefore") do not.** Flat across correct and wrong
  votes in all six cells, and non-monotonic across models. Causal-connective language is
  boilerplate here.
- **An analyst-authored "night-order vocabulary" failed outright** -- three of its terms
  matched **zero** times in 2,287 justifications, and the category cannot separate
  *relaying another player's claim* from *using a rule as a premise*. Retained as
  explicitly exploratory, with a coverage diagnostic that auto-flags any category under
  20% coverage.

Note the pattern: the marker class that works is the one **semantically aligned with the
capability being measured** -- the questionnaire independently identifies *temporal
state-tracking* as the discriminating skill, and it is *temporal* discourse language that
separates the models, while *causal* discourse language does not.

The role of ground-truth verification (Analyses 5-6) is therefore to **validate** text-based
measures rather than replace them: it is what established that Temporal density was
tracking something real and Contingency density was not.

## Multiplicity: how many of these results could be chance?

This notebook reports roughly **120 separate 95% confidence intervals** with **no
multiplicity correction**, so about **6 false positives are expected by chance alone**.
That does not threaten the large effects -- Analysis 5's cross-model contrasts have CIs
like [0.262, 0.456], nowhere near the boundary, and would survive any correction. It does
matter for marginal ones:

- **Analysis 2's single significant cell** (4B stochastic, Contingency, CI
  [-1.439, -0.031]) barely clears zero and is **1 hit out of 12 tests** -- exactly the
  chance rate. It should be treated as a **probable false positive**, not a finding. The
  honest reading of Analysis 2 is that it is a clean null throughout, which is a *simpler*
  and stronger statement than "null with one unexplained exception".
- Any cell whose CI only just excludes zero should be read the same way.

The general rule for this notebook: **effect size and n matter more than the binary
`excludes_zero` flag.** Cross-check every claim against the reported `n_games`/
`n_instances` and the coverage diagnostics before treating it as real.

---

## Reading guide: which contrasts are conclusive at this n

- Every bootstrap table above carries `excludes_zero` / `inconclusive_at_this_n`
  per row -- only rows with `excludes_zero == True` should be narrated as a
  real difference in the write-up.
- **`excludes_zero` is necessary but not sufficient.** Always cross-check a
  contrast against its reported `n_games` / `n_instances`. A narrow CI on a
  thin subset (e.g. Analysis 6's 13-22 games per cell) is not a finding.
- **Judgment call:** `most_accused_player` ties are broken alphabetically
  (deterministic) rather than randomly, unlike the surrogate notebook's own
  baseline -- reasonable for a descriptive analysis, but worth knowing if
  results are compared directly against the surrogate's `most_accused_ww`
  baseline numbers.
- **Analysis 5 caveat:** `share_both` (mentions both the start and end role
  for a swap-target vote) is the intended headline number for "does the
  bigger model deduce rule-based swaps more" -- but co-occurrence is not
  proof of assertion. The DiMLex-connective check raises confidence when it
  also clears its own CI, but neither check certifies causal faithfulness.
- **Analysis 6 caveat:** ground truth here is DeepSeek-annotated
  (`is_self_contradiction`/`roles_claimed`), not human-verified like
  Analysis 5's `start_roles`/`end_roles`. It is also a rarer diagnostic
  subset than Analysis 5's swap targets (check `n_games`/`n_instances` in
  `06a_contradiction_flagging_bootstrap_ci.csv` before treating any contrast
  as more than exploratory).
- **Chance-adjusted checks (5 & 6):** `mean_excess_both` /
  `mean_excess_flagged` are the numbers that actually answer "is this
  targeted, or just a verbose model naming lots of roles" -- read these
  alongside, not instead of, the raw `share_both`/`share_flagged` numbers.
- **Analysis 7 caveat:** this reframes "crowd persuasion" as "shared
  susceptibility to the same social-salience signal" -- it cannot show
  causation either way, only that herded and independent votes differ in how
  well the salience proxy explains them. A per-model gap (rather than a flat
  effect) is the interesting result here, not a single pooled number.
- **Weighted companions (2w, 7w):** `contrast_table`'s hard classification
  gives a game with a 2-1 stochastic split full weight in both arms of a
  contrast; `weighted_contrast_table` instead weights each arm by that
  game's actual vote share. The two can disagree, and when they do, the
  weighted version is the more defensible one for a T=1-sampling design --
  but both are kept, since the hard-classification numbers are already
  reported elsewhere in this analysis.

In [29]:
excludes_zero_summary = pd.concat([
    surrogate_contingency_ci.assign(analysis="1_surrogate_predictability")[["analysis", "model", "decoding_group", "excludes_zero"]],
    markers_by_herded_independent_ci.assign(analysis="2_herded_vs_independent")[["analysis", "model", "decoding_group", "marker_category", "excludes_zero"]],
    swap_target_bootstrap_ci.assign(analysis="5_swap_target_role_attribution")[["analysis", "decoding_group", "model_a", "model_b", "excludes_zero"]],
    swap_chance_adjusted_check.assign(analysis="5c_chance_adjusted")[["analysis", "model", "decoding_group", "excludes_zero"]],
    contradiction_bootstrap_ci.assign(analysis="6_contradiction_flagging")[["analysis", "decoding_group", "model_a", "model_b", "excludes_zero"]],
    contradiction_chance_adjusted_check.assign(analysis="6c_chance_adjusted")[["analysis", "model", "decoding_group", "excludes_zero"]],
    salience_reliance_gap_ci.assign(analysis="7_salience_reliance_gap")[["analysis", "model", "decoding_group", "excludes_zero"]],
    markers_by_herded_independent_weighted_ci.assign(analysis="2w_herded_vs_independent_weighted")[["analysis", "model", "decoding_group", "marker_category", "excludes_zero"]],
    salience_reliance_gap_weighted_ci.assign(analysis="7w_salience_reliance_gap_weighted")[["analysis", "model", "decoding_group", "excludes_zero"]],
    salience_baseline_corrected_ci.assign(analysis="7b_salience_baseline_corrected")[["analysis", "model", "decoding_group", "excludes_zero"]],
], ignore_index=True)
print(excludes_zero_summary.to_string())

                             analysis model decoding_group  excludes_zero marker_category model_a model_b
0          1_surrogate_predictability    2B         Greedy           True             NaN     NaN     NaN
1          1_surrogate_predictability    2B     Stochastic          False             NaN     NaN     NaN
2          1_surrogate_predictability   31B         Greedy           True             NaN     NaN     NaN
3          1_surrogate_predictability   31B     Stochastic          False             NaN     NaN     NaN
4          1_surrogate_predictability    4B         Greedy          False             NaN     NaN     NaN
5          1_surrogate_predictability    4B     Stochastic          False             NaN     NaN     NaN
6             2_herded_vs_independent    2B         Greedy          False     Contingency     NaN     NaN
7             2_herded_vs_independent    2B     Stochastic          False     Contingency     NaN     NaN
8             2_herded_vs_independent   31B   